# Подбор порога для неполных дублей FAQ по E5

Ноутбук получает соседей по cosine готовых эмбеддингов ответов. После поиска проверяет контекст пары: продукт, операцию, числа, запреты и ссылки. Полные совпадения попадают на отдельную вкладку. Итог — кандидаты для редакторской проверки.

**Вход** — две выгрузки FAQ, полный `faq_embeddings_final.npy` и снимок ID и ответов в порядке строк того же запуска. Укажите пути ниже и выберите **Run All**. Для выполнения нужны Python 3.10+, `numpy`, `pandas` и `openpyxl`.
**Для автоматического подбора порога** в ноутбук включены отпечатки и метки 473 ранее размеченных пар. При наличии `AUDIT_FILE` используется указанный там JSON.

In [ ]:
from pathlib import Path

ROOT = Path.cwd()
EXPORT_FILES = [
    ROOT / "data" / "export_faq_idp_main_elch.xlsx",
    ROOT / "data" / "export_faq_idp_additional_elch.xlsx",
]
E5_NPY = ROOT / "faq_embeddings_final.npy"
# Доверенный локальный PKL или CSV со столбцами «Идентификатор» и «Ответ».
E5_SNAPSHOT = ROOT / "faq_answers_snapshot.pkl"
AUDIT_FILE = ROOT / "faq_e5_audit.json"
BENCHMARK_FILE = ROOT / "faq_partial_benchmark.csv"
if not BENCHMARK_FILE.is_file():
    BENCHMARK_FILE = ROOT / "results" / "faq_partial_benchmark.csv"
OUTPUT_DIR = ROOT / "results" / "e5_run"

TOP_K = 12
REQUIRE_FULL_COVERAGE = False

## Проверка и результаты

В поиск попадают карточки, у которых совпали ID и текст ответа в кэше и в текущей выгрузке. При частичном покрытии выводится число карточек без вектора. Косинус рассчитывается блоками из уже сохранённых векторов. `TOP_K` действует отдельно для соседей из того же продукта и из других продуктов.

Ноутбук сопоставляет ранее вычитанные пары по отпечаткам карточек. Сравнивает cosine E5, проверки условий и добавление направленной вложенности. На части «подбор» выбираются порог и правило по F1. Часть «проверка» используется для оценки выбранного порога. Неоднозначные метки исключаются. В варианте с вложенностью паре со строгой текстовой вложенностью или высоким направленным покрытием разрешается не дотянуть до базового порога на 0,02.

В первом запуске достаточно вывода последней ячейки. Для сохранения HTML и Excel в `OUTPUT_DIR` включите `WRITE_REPORT = True`. Метрики относятся к размеченной выборке. Для оценки работы RAG нужна проверка на реальных поисковых запросах.

Для обсуждения порога можно сфотографировать вывод двух последних ячеек. Таблица «Объём по порогам» показывает, сколько пар проходит каждый порог до проверки условий. Количество пар само по себе не показывает качество. В отчёте с разметкой сравнивайте число найденных полезных пар и число лишних.

In [ ]:
from dataclasses import dataclass
from collections import Counter
import re
from openpyxl import load_workbook

CONTEXT_FIELDS = ('Продукт', 'Подпродукт', 'Тематика', 'Основной канал',
                  'Дополнительный канал', 'Дата, время включения', 'Дата, время отключения')
MAIN, EXTRA = 'Основной канал', 'Дополнительный канал'
LEGACY = 'Перенесено из предыдущего отчёта'
SAME, DIFFERENT, UNKNOWN = 'same', 'different', 'unknown'
INSURANCE_PROGRAM = re.compile(r'\bзащит[аеуы]\s+(дома|денег)\b', re.I)
DEPOSIT_CLOSE = re.compile(r'^как\s+(?:досрочно\s+)?закрыть\s+вклад\s+[«"]([^»"]+)[»"]\s*\??$', re.I)
NUMBERS = re.compile(r'\d{1,3}(?:[ \u00a0]\d{3})+(?:[.,]\d+)?|\d+(?:[.,:/-]\d+)*')
NEGATION = re.compile(r'\b(?:не|нет|нельзя|без)\b', re.I)


def clean(value):
    return '' if value is None else str(value)


def norm(value):
    return ' '.join(clean(value).split())


def numbers(answer):
    return {re.sub(r'\s', '', token).replace(',', '.')
            for token in NUMBERS.findall(answer)}


def differences(a, b):
    links = {field for field in a.row.keys() | b.row.keys() if field and field.startswith('Ссылка')}
    fields = [field for field in (*CONTEXT_FIELDS, *sorted(links))
              if norm(a.row.get(field)) != norm(b.row.get(field))]
    if numbers(a.answer) != numbers(b.answer):
        fields.append('Числа в тексте')
    if Counter(NEGATION.findall(a.answer.casefold())) != Counter(NEGATION.findall(b.answer.casefold())):
        fields.append('Отрицания в тексте')
    return fields


@dataclass(frozen=True)
class FAQ:
    id: str
    question: str
    answer: str
    alternatives: tuple
    source: str
    row: dict

    @property
    def product_key(self):
        return tuple(norm(self.row.get(field)).casefold() for field in CONTEXT_FIELDS[:2])


def context(a, b):
    ap, asub = a.product_key
    bp, bsub = b.product_key
    if ap and bp and ap != bp:
        return DIFFERENT, 'Разные продукты'
    if not ap or not bp or not asub or not bsub:
        return UNKNOWN, 'У одной или обеих карточек не заполнен продукт или подпродукт'
    if asub != bsub:
        return DIFFERENT, 'Разные подпродукты'
    program = None
    if (ap, asub) == ('страхование', 'злс'):
        program = INSURANCE_PROGRAM
    elif ap == 'вклады и счета':
        program = DEPOSIT_CLOSE
    if program:
        left, right = (program.search(norm(card.question)) for card in (a, b))
        if left and right and norm(left[1]).casefold() != norm(right[1]).casefold():
            return DIFFERENT, 'Разные программы внутри подпродукта'
    return SAME, 'Совпадают продукт и подпродукт'


def read_records(paths):
    records, seen = {}, set()
    counts = Counter()
    required = {'Идентификатор', 'Вопрос', 'Ответ', 'Продукт', 'Подпродукт', 'Статус', MAIN, EXTRA}
    for path in paths:
        book = load_workbook(path, read_only=True, data_only=True)
        try:
            rows = iter(book['Выгрузка'].values)
            fields = next(rows)
            if missing := required.difference(fields):
                raise ValueError(f'{path.name}: отсутствуют поля {sorted(missing)}')
            alt_fields = [f for f in fields if f and f.startswith('Альтернативный вопрос')]
            for line, values in enumerate(rows, 2):
                if all(value is None for value in values):
                    continue
                counts['Строк в выгрузках'] += 1
                row = dict(zip(fields, values))
                ident = norm(row['Идентификатор'])
                if not ident or ident in seen:
                    raise ValueError(f'{path.name}:{line}: пустой или повторяющийся ID FAQ')
                seen.add(ident)
                main, extra = ('ЕЛЧ' in {norm(x) for x in clean(row[key]).split(',')}
                               for key in (MAIN, EXTRA))
                if norm(row['Статус']) != 'Активно' or not (main or extra):
                    continue
                if not norm(row['Вопрос']) or not norm(row['Ответ']):
                    raise ValueError(f'{path.name}:{line}: пустой вопрос или ответ')
                source = 'main' if main else 'additional'
                counts[source] += 1
                records[ident] = FAQ(ident, clean(row['Вопрос']), clean(row['Ответ']),
                                     tuple(clean(row[f]) for f in alt_fields if norm(row[f])),
                                     source, row)
        finally:
            book.close()
    return records, counts

In [ ]:
"""Lightweight, offline candidate search for incomplete FAQ duplicates."""

from collections import Counter, defaultdict
from dataclasses import dataclass
import math
from pathlib import Path
import re
import unicodedata

import numpy as np


TOKEN = re.compile(r'[^\W\d_]+|\d+(?:[.,:/-]\d+)*', re.UNICODE)
LIST_NUMBER = re.compile(r'(?m)^\s*\d{1,2}[.)]\s+')
RISK_NUMBER = re.compile(
    r'(?<!\w)(\d+(?:[ \u00a0]\d{3})*(?:[.,]\d+)?)\s*'
    r'(%|рабоч\w*\s+(?:дн\w*|день)|календар\w*\s+(?:дн\w*|день)|дн(?:я|ей)?\b|день\b|'
    r'руб\w*|₽|тыс\w*|млн\w*|мес\w*|год\w*|лет\b|час\w*)', re.I)
ARTICLE = re.compile(r'\b(статья|пункт|подпункт)\s*(\d+(?:\.\d+)*)', re.I)
QUOTES = re.compile(r'[«"]([^»"]+)[»"]')
NAMED_DEPOSIT = re.compile(r'\b(?:вклад|сч[её]т)\w*\s*[«"]([^»"]+)[»"]', re.I)
PRIME = re.compile(r'\bсберпрайм\b(?:\s*(?:\+|плюс))?', re.I)
SCOPE = re.compile(r'\b(?:сберперв\w*|сберпремьер\w*|доступн\w* выгод\w*|свой максимум|'
                   r'подушка безопасности|страхование по часам|дмс|защита от травм)\b')
URL = re.compile(r'https?://\S+', re.I)
QUOTE_CHARS = str.maketrans('', '', '«»„“”"')
DASH_CHARS = str.maketrans({'—': '–', '−': '–'})
OPERATIONS = [
    (r'\b(?:внос\w*|внести|полож(?!ен)\w*|пополн\w*)\b', r'\b(?:снят\w*|сним\w*|выдал|выдач\w*)\b', 'Внесение и выдача'),
    (r'\bподключ\w*\b', r'\bотключ\w*\b', 'Подключение и отключение'),
    (r'\bдобав\w*\b', r'\bудал\w*\b', 'Добавление и удаление'),
    (r'\bоткр\w*\b', r'\bзакр\w*\b', 'Открытие и закрытие'),
]
OPERATIONS = [(re.compile(a), re.compile(b), label) for a, b, label in OPERATIONS]
STRONG_NEGATION = re.compile(r'\b(?:нельзя|запрещ\w*|не\s+допуска\w*|не\s+предусмотр\w*)\b')




def words(text):
    return TOKEN.findall(LIST_NUMBER.sub('', text).casefold().replace('ё', 'е'))


def answer_key(text):
    # Preserve units, +, digits, punctuation inside numbers, and case-sensitive URLs.
    if re.search(r'https?://', text, re.I):
        return norm(text)
    normalized = unicodedata.normalize('NFC', text).translate(QUOTE_CHARS).translate(DASH_CHARS)
    return norm(normalized.casefold().replace('ё', 'е')).removesuffix('.').rstrip()


def weighted_matrix(documents, max_df=.20):
    """Return unit TF-IDF rows for unigrams and two-word phrases."""
    from scipy.sparse import csr_matrix

    count = len(documents)
    terms = [Counter([*tokens, *zip(tokens, tokens[1:])]) for tokens in documents]
    frequency = Counter(term for row in terms for term in row)
    vocab = {term: k for k, term in enumerate(
        term for term, df in frequency.items() if df <= max(2, count * max_df))}
    indptr, indices, data = [0], [], []
    for row in terms:
        weighted = [(vocab[t], (1 + math.log(n)) * (1 + math.log((1 + count) / (1 + frequency[t]))))
                    for t, n in row.items() if t in vocab]
        size = math.sqrt(sum(w * w for _, w in weighted))
        indices.extend(i for i, _ in weighted)
        data.extend(w / size for _, w in weighted)
        indptr.append(len(data))
    result = csr_matrix((np.asarray(data, dtype=np.float32), indices, indptr),
                        shape=(count, len(vocab)), dtype=np.float32)
    result.sort_indices()
    return result


def embedding_cosines(pairs, dense):
    scores = [None] * len(pairs)
    if dense is None:
        return scores
    vectors, positions = dense
    hits = [(k, positions[a], positions[b]) for k, (a, b) in enumerate(pairs)
            if a in positions and b in positions]
    if hits:
        indices, left, right = np.asarray(hits).T
        values = np.einsum('ij,ij->i', vectors[left], vectors[right]).clip(-1, 1)
        for index, value in zip(indices, values):
            scores[index] = float(value)
    return scores


class Corpus:
    def __init__(self, records, *, lexical=True):
        self.cards = [records[key] for key in sorted(records)]
        self.index = {card.id: i for i, card in enumerate(self.cards)}
        self.form = [answer_key(c.answer) for c in self.cards]
        self.answer_matrix = self.question_matrix = None
        answers = [words(c.answer) for c in self.cards]
        if lexical:
            questions = [words(c.question) for c in self.cards]
            self.answer_matrix = weighted_matrix(answers)
            self.question_matrix = weighted_matrix(questions, max_df=.40)
        counts = Counter(w for tokens in answers for w in set(tokens))
        self.idf = {w: 1 + math.log((1 + len(answers)) / (1 + n)) for w, n in counts.items()}
        self.tokens = [Counter(tokens) for tokens in answers]
        self.token_text = [' ' + ' '.join(tokens) + ' ' for tokens in answers]
        self.token_count = list(map(len, answers))
        self.weight = [sum(self.idf[w] * n for w, n in ts.items()) for ts in self.tokens]
        self.question_forms = [{norm(q).casefold() for q in (c.question, *c.alternatives)}
                               for c in self.cards]
        self.product = np.array([c.product_key[0] for c in self.cards], dtype=object)

    def pool(self, top_k=12, block=96, answer_floor=.36, question_floor=.58, dense=None,
             dense_search_floor=None):
        """Keep independent neighbour budgets inside and across products for each signal."""
        if top_k < 1 or block < 1:
            raise ValueError('Размер блока и число соседей должны быть положительными.')
        seen = defaultdict(set)
        forms = np.asarray(self.form, dtype=object)

        def select(row, columns, scores, tag):
            if not columns.size:
                return
            eligible = (columns != row) & (forms[columns] != forms[row])
            same = (self.product[columns] == self.product[row]) & bool(self.product[row])
            for mask in (eligible & same, eligible & ~same):
                offsets = np.nonzero(mask)[0]
                if offsets.size > top_k:
                    border = np.partition(scores[offsets], -top_k)[-top_k]
                    offsets = offsets[scores[offsets] >= border]
                    order = np.lexsort((columns[offsets], -scores[offsets]))[:top_k]
                    offsets = offsets[order]
                for other in columns[offsets]:
                    seen[tuple(sorted((row, int(other))))].add(tag)

        if self.answer_matrix is None and dense_search_floor is None:
            raise ValueError('Для поиска E5 нужны готовые векторы и порог поиска соседей.')
        channels = ((self.answer_matrix, answer_floor, 'ответы'),
                    (self.question_matrix, question_floor, 'вопросы'))
        for matrix, floor, tag in channels:
            if matrix is None:
                continue
            transpose = matrix.T
            for start in range(0, len(self.cards), block):
                batch = (matrix[start:start + block] @ transpose).tocsr()
                for offset in range(batch.shape[0]):
                    lo, hi = batch.indptr[offset:offset + 2]
                    columns, scores = batch.indices[lo:hi], batch.data[lo:hi]
                    good = scores >= floor
                    select(start + offset, columns[good], scores[good], tag)

        if self.question_matrix is not None:
            # Alternative-question overlaps supplement lexical neighbours only.
            inverted = defaultdict(list)
            for row, forms in enumerate(self.question_forms):
                for form in forms:
                    if len(words(form)) >= 3:
                        inverted[form].append(row)
            for rows in inverted.values():
                if len(rows) > 24:
                    continue
                for n, i in enumerate(rows):
                    for j in rows[n + 1:]:
                        if self.form[i] != self.form[j]:
                            seen[i, j].add('та же формулировка вопроса')

        if dense_search_floor is not None:
            if dense is None:
                raise ValueError('Для поиска соседей E5 нужен проверенный кэш.')
            vectors, positions = dense
            keys = np.array(sorted(positions), dtype=int)
            if keys.size:
                sample = vectors[[positions[int(i)] for i in keys]]
                for start in range(0, len(keys), block):
                    scores = sample[start:start + block] @ sample.T
                    for offset, score in enumerate(scores):
                        row = int(keys[start + offset])
                        keep = score >= dense_search_floor
                        select(row, keys[keep], score[keep], 'E5')
        return seen

    def features(self, pairs, dense=None):
        keys = sorted(pairs)
        for start in range(0, len(keys), 1024):
            batch = keys[start:start + 1024]
            if self.answer_matrix is not None:
                ai, bi = np.asarray(batch).T
                score_a = np.asarray(self.answer_matrix[ai].multiply(self.answer_matrix[bi]).sum(axis=1)).ravel()
                score_q = np.asarray(self.question_matrix[ai].multiply(self.question_matrix[bi]).sum(axis=1)).ravel()
            else:
                score_a = score_q = [None] * len(batch)
            for (i, j), a_score, q_score, dense_score in zip(batch, score_a, score_q, embedding_cosines(batch, dense)):
                a, b = self.cards[i], self.cards[j]
                shared = sum(self.idf[w] * n for w, n in (self.tokens[i] & self.tokens[j]).items())
                cov_a = shared / self.weight[i] if self.weight[i] else 0
                cov_b = shared / self.weight[j] if self.weight[j] else 0
                same_question = bool(self.question_forms[i] & self.question_forms[j])
                score = {
                    'a': a.id, 'b': b.id, 'answer': None if a_score is None else min(1., float(a_score)),
                    'question': None if q_score is None else min(1., float(q_score)), 'same_question': same_question,
                    'cover_a': min(1., cov_a), 'cover_b': min(1., cov_b), 'embedding': dense_score,
                    'literal': min(self.token_count[i], self.token_count[j]) >= 8
                    and (self.token_text[i] in self.token_text[j] or self.token_text[j] in self.token_text[i]),
                }
                yield (i, j), score


def _numbers_with_units(text):
    result = defaultdict(set)
    for value, unit in RISK_NUMBER.findall(text.casefold()):
        unit = norm(unit)
        if unit.startswith(('рабоч', 'календар')):
            unit = unit.split()[0][:5] + ' дни'
        elif unit.startswith(('дн', 'ден')):
            unit = 'дни'
        elif unit == 'лет':
            unit = 'год'
        else:
            unit = unit[:3]
        result[unit].add(float(value.replace(' ', '').replace('\u00a0', '').replace(',', '.')))
    return result


def risks(a, b):
    relation, subtype = context(a, b)
    qa, qb = [norm(x.question).casefold().replace('ё', 'е') for x in (a, b)]
    qa_refs = {(k.casefold(), n) for k, n in ARTICLE.findall(qa)}
    qb_refs = {(k.casefold(), n) for k, n in ARTICLE.findall(qb)}
    hard, caution = [], []
    same_product = bool(a.product_key[0] and a.product_key[0] == b.product_key[0])
    if relation == 'different' and 'внутри' in subtype:
        hard.append(subtype)
    elif relation != 'same':
        caution.append(subtype)
    if any({n for k2, n in qa_refs if k2 == k}.isdisjoint({n for k2, n in qb_refs if k2 == k})
           for k in set(k for k, _ in qa_refs) & set(k for k, _ in qb_refs)):
        hard.append('Разные статьи или пункты')
    names_a = {norm(s) for s in QUOTES.findall(qa)}
    names_b = {norm(s) for s in QUOTES.findall(qb)}
    if same_product and a.product_key[0] == 'вклады и счета':
        names_a = names_a or {norm(s) for s in NAMED_DEPOSIT.findall(a.answer[:160].casefold())}
        names_b = names_b or {norm(s) for s in NAMED_DEPOSIT.findall(b.answer[:160].casefold())}
    prime_a, prime_b = PRIME.search(qa), PRIME.search(qb)
    if prime_a and prime_b and (prime_a[0] == 'сберпрайм') != (prime_b[0] == 'сберпрайм'):
        hard.append('Разные пакеты СберПрайм')
    scope_a, scope_b = set(SCOPE.findall(qa)), set(SCOPE.findall(qb))
    if scope_a and scope_b and scope_a.isdisjoint(scope_b):
        caution.append('Отличается явно названная программа или пакет')
    named_scope = a.product_key[0] in ('вклады и счета', 'страхование', 'инвестиции и ценные бумаги ук')
    if same_product and named_scope and names_a and names_b and names_a.isdisjoint(names_b):
        hard.append('Разные явные названия продукта или программы')
    for x, y, reason in OPERATIONS:
        a_action, b_action = (bool(x.search(qa)), bool(y.search(qa))), (bool(x.search(qb)), bool(y.search(qb)))
        if (a_action, b_action) in (((True, False), (False, True)), ((False, True), (True, False))):
            hard.append('Разные операции: ' + reason)
    na, nb = _numbers_with_units(a.answer), _numbers_with_units(b.answer)
    for unit in sorted(na.keys() & nb.keys()):
        if na[unit].isdisjoint(nb[unit]):
            hard.append(f'Разные значения ({unit}): {sorted(na[unit])} ↔ {sorted(nb[unit])}')
        elif na[unit] != nb[unit]:
            caution.append(f'Дополнительные числовые условия ({unit})')
    if bool(STRONG_NEGATION.search(a.answer.casefold())) != bool(STRONG_NEGATION.search(b.answer.casefold())):
        caution.append('Явный запрет только в одном ответе')
    if (set(URL.findall(a.answer)) != set(URL.findall(b.answer))
            or any(flag.startswith('Ссылка') for flag in differences(a, b))):
        caution.append('Разные ссылки')
    return same_product, relation, hard, caution




def load_embedding_cache(path, snapshot, corpus, *, expected_dimension=None, require_full_coverage=False):
    """Match row positions from the original run to current cards by ID and answer text."""
    import pandas as pd

    vectors = np.load(path, mmap_mode='r', allow_pickle=False)
    if vectors.ndim != 2 or vectors.shape[1] < 1 or vectors.dtype.kind != 'f':
        raise ValueError('Ожидается двумерный массив векторов с плавающей точкой.')
    if expected_dimension is not None and vectors.shape[1] != expected_dimension:
        raise ValueError(f'Ожидалось {expected_dimension} признаков в векторе, получено {vectors.shape[1]}.')
    if Path(snapshot).suffix.lower() == '.csv':
        frame = pd.read_csv(snapshot, dtype=str, keep_default_na=False, encoding='utf-8-sig')
    elif Path(snapshot).suffix.lower() == '.pkl':
        frame = pd.read_pickle(snapshot)
    else:
        raise ValueError('Доступны CSV со столбцами Идентификатор и Ответ или доверенный локальный PKL.')
    ids = frame['Идентификатор'].fillna('').astype(str).str.strip()
    answers = frame['Ответ'].fillna('').astype(str).str.strip()
    if ids.empty or ids.eq('').any() or ids.duplicated().any() or answers.eq('').any():
        raise ValueError('В снимке кэша пустой ответ или пустой/повторный ID.')
    if vectors.shape[0] != len(ids):
        raise ValueError('Число строк в NPY и снимке разное.')
    matches = [(i, corpus.index[k]) for i, (k, a) in enumerate(zip(ids, answers))
               if k in corpus.index and corpus.cards[corpus.index[k]].answer.strip() == a]
    if not matches:
        raise ValueError('Ни одного совпадения ID и ответа между выгрузками и снимком.')
    if require_full_coverage and len(matches) != len(corpus.cards):
        raise ValueError(f'Совпали ID и ответы у {len(matches)} из {len(corpus.cards)} FAQ. '
                         'Для полного отчёта нужны векторы для текущей выгрузки.')
    a, b = np.asarray(matches).T
    selected = vectors[a].astype(np.float32)
    lengths = np.linalg.norm(selected, axis=1)
    if not np.isfinite(lengths).all() or (lengths == 0).any():
        raise ValueError('Кэш содержит нулевые или нечисловые векторы среди совпавших FAQ.')
    selected /= lengths[:, None]
    return selected, dict(zip(b.tolist(), range(len(b))))

In [ ]:
"""Benchmark validation, queue construction, and offline reports for FAQ review."""

from collections import Counter, defaultdict
import csv
from hashlib import sha256
from html import escape
import json
from pathlib import Path
from tempfile import TemporaryDirectory

from openpyxl import Workbook
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter



POSITIVE = {'дубль', 'покрытие'}
REVIEW = {'приоритетная вычитка', 'сверка контекста'}
SEVERITY = {'приоритетная вычитка': 0, 'сверка контекста': 1, 'существенное различие': 2}


def fingerprint(card):
    payload = [card.question, *card.alternatives, card.answer, *card.product_key]
    return sha256(json.dumps(payload, ensure_ascii=False).encode()).hexdigest()


def load_benchmark(path, corpus):
    if path is None:
        return {}, 0
    with Path(path).open(encoding='utf-8-sig', newline='') as stream:
        rows = list(csv.DictReader(stream))
    good, stale = {}, 0
    for row in rows:
        a, b = row['id_a'], row['id_b']
        if (a not in corpus.index or b not in corpus.index
                or fingerprint(corpus.cards[corpus.index[a]]) != row['hash_a']
                or fingerprint(corpus.cards[corpus.index[b]]) != row['hash_b']):
            stale += 1
            continue
        key = tuple(sorted((corpus.index[a], corpus.index[b])))
        if key in good:
            raise ValueError(f'Повторная разметка пары {a} и {b}.')
        good[key] = row
    return good, stale










def exact_report(corpus):
    grouped = defaultdict(list)
    for k, card in zip(corpus.form, corpus.cards):
        grouped[k].append(card)
    rows = []
    for cards in grouped.values():
        if len(cards) < 2:
            continue
        for card in cards:
            rows.append((len(rows) + 1, cards[0].id, len(cards), card.id, card.row['Продукт'],
                         card.row['Подпродукт'] or '', card.question, card.answer))
    return rows


def view_row(r):
    a, b = r['cards']
    same, relation, hard, soft = r['flags']
    audit = r['audit']
    queue = r['queue']
    if not r['sources']:
        queue = 'вне пула соседей'
    if answer_key(a.answer) == answer_key(b.answer):
        queue = 'совпадает после нормализации'
    return (queue, 'один продукт' if same else 'между продуктами или продукт пуст',
            r['answer'], r['question'], float(r['cover_a']), float(r['cover_b']), r['embedding'],
            r['signals'], ' | '.join([*hard, *soft]), audit.get('decision', ''),
            audit.get('comment', ''), audit.get('old_verdict', ''), a.id, b.id,
            a.row['Продукт'], a.row['Подпродукт'] or '', b.row['Продукт'], b.row['Подпродукт'] or '',
            a.question, b.question, a.answer, b.answer, ' | '.join(r['sources']), audit.get('source', ''))


PAIR_HEADER = ('Очередь', 'Сравнение', 'Cosine ответа (TF-IDF)', 'Cosine вопроса (TF-IDF)',
               'Доля A в B', 'Доля B в A', 'Cosine E5', 'Сигналы', 'Что проверить',
               'Решение по вычитанной паре', 'Комментарий', 'Старый вердикт ГигаЧата',
               'ID A', 'ID B', 'Продукт A', 'Подпродукт A', 'Продукт B', 'Подпродукт B',
               'Вопрос A', 'Вопрос B', 'Ответ A', 'Ответ B', 'Источник поиска', 'Источник разметки')
EXACT_HEADER = ('Строка', 'Группа', 'Размер', 'ID FAQ', 'Продукт', 'Подпродукт', 'Вопрос', 'Ответ')
METRIC_HEADER = ('Раздел', 'Метод', 'Размеченных пар', 'Найдено полезных', 'Лишних', 'Пропущено полезных',
                 'Доля полезных среди найденных', 'Доля найденных среди полезных в выборке')


def write_workbook(path, summaries, candidates, differences, audited, exact, metrics, lexical, sweep,
                   *, e5=False, pool_counts=()):
    wb = Workbook()
    wb.remove(wb.active)

    def sheet(name, header, data, widths, score_cols=(), row_height=34):
        ws = wb.create_sheet(name)
        ws.append(header)
        for row in data:
            ws.append(row)
            for cell in ws[ws.max_row]:
                if isinstance(cell.value, str):
                    cell.data_type = 's'
            ws.row_dimensions[ws.max_row].height = row_height
        ws.freeze_panes = 'A2'
        ws.auto_filter.ref = ws.dimensions
        ws.sheet_view.showGridLines = False
        ws.row_dimensions[1].height = 34
        for cell in ws[1]:
            cell.fill = PatternFill('solid', fgColor='233F53')
            cell.font = Font(bold=True, color='FFFFFF')
            cell.alignment = Alignment(vertical='center', wrap_text=True)
        for n, width in enumerate(widths, 1):
            ws.column_dimensions[get_column_letter(n)].width = width
        for row in ws.iter_rows(min_row=2):
            for cell in row:
                if cell.column in score_cols:
                    cell.number_format = '0.000'
                cell.alignment = Alignment(vertical='top', wrap_text=True)
        return ws

    sheet('Сводка', ('Показатель', 'Значение'), summaries, (43, 98))
    sizes = (23, 23, 18, 18, 13, 13, 13, 32, 52, 20, 50, 53,
             34, 34, 24, 24, 24, 24, 47, 47, 65, 65, 26, 24)
    columns = [i for i in range(len(PAIR_HEADER)) if not e5 or i not in (2, 3)]
    header = tuple(PAIR_HEADER[i] for i in columns)
    widths = tuple(sizes[i] for i in columns)
    score_cols = tuple(n for n, i in enumerate(columns, 1) if 2 <= i <= 6)
    for name, rows, height in (('Неполные дубли', candidates, 70),
                               ('Существенные различия', differences, 58),
                               ('Аудит и разметка', audited, 68)):
        values = (tuple(row[i] for i in columns) for row in map(view_row, rows))
        sheet(name, header, values, widths, score_cols, height)
    sheet('Полные совпадения', EXACT_HEADER, exact, (12, 35, 12, 35, 24, 25, 53, 86), (), 46)
    sheet('Проверка порогов', METRIC_HEADER, metrics, (18, 39, 16, 18, 14, 21, 24, 24), (7, 8))
    if lexical:
        sheet('Сетка TF-IDF', ('Порог ответа', 'Порог вопроса', 'Порог покрытия', *METRIC_HEADER[2:]),
              lexical, (16, 16, 17, 16, 18, 14, 21, 24, 24), (1, 2, 3, 8, 9))
    if sweep:
        sheet('Пороги E5', ('Порог E5', *METRIC_HEADER), sweep,
              (14, 18, 39, 16, 18, 14, 21, 24, 24), (1, 8, 9))
    if pool_counts:
        sheet('Объём по порогам', ('Порог E5', 'Внутри продукта', 'Между продуктами или продукт пуст'),
              pool_counts, (15, 21, 36), (1,))
    wb.save(path)


def write_html(path, summaries, candidates, differences, exact, metrics, *, e5=False):
    def esc(x):
        return escape(str(x), quote=True)

    def pair_table(rows):
        score_header = 'Cosine E5' if e5 else 'Cosine ответа / вопроса'
        out = ['<table><thead><tr><th>Пара и ответы</th><th>Очередь</th>'
               f'<th>{score_header}</th><th>Покрытие A / B</th><th>Замечания</th></tr></thead><tbody>']
        for r in rows:
            a, b = r['cards']
            same, _, hard, soft = r['flags']
            audit = r['audit']
            ids = esc(a.id + ' · ' + b.id)
            product = esc(' / '.join((a.row['Продукт'] or '—', b.row['Продукт'] or '—')))
            notes = ' · '.join(filter(None, [*hard, *soft, audit.get('comment', '')]))
            cosine = f'{r["embedding"]:.3f}' if e5 else f'{r["answer"]:.3f} / {r["question"]:.3f}'
            if not e5 and r['embedding'] is not None:
                cosine += f'<br>E5 {r["embedding"]:.3f}'
            info = esc(audit.get('decision', ''))
            out.append(
                f'<tr data-scope="{int(same)}" data-queue="{esc(r["queue"])}"><td><small>{product}</small>'
                f'<details><summary>{esc(a.question)}<br>{esc(b.question)}</summary>'
                f'<p><small>{ids}</small></p><p><b>Ответ A</b><br>{esc(a.answer)}</p>'
                f'<p><b>Ответ B</b><br>{esc(b.answer)}</p></details></td>'
                f'<td>{esc(r["queue"])}<br><small>{info}</small></td>'
                f'<td>{cosine}</td>'
                f'<td>{r["cover_a"]:.3f} / {r["cover_b"]:.3f}</td><td>{esc(notes)}</td></tr>')
        out.append('</tbody></table>')
        return ''.join(out)

    def simple_table(header, rows, attrs=False):
        return ('<table><thead><tr>' + ''.join(f'<th>{esc(x)}</th>' for x in header) +
                '</tr></thead><tbody>' + ''.join(
                    '<tr' + (' data-scope="" data-queue=""' if attrs else '') + '>' + ''.join(
                        f'<td>{esc(x if x is not None else "—")}</td>' for x in row) + '</tr>' for row in rows) +
                '</tbody></table>')

    body = ['<!doctype html><html lang="ru"><meta charset="utf-8"><meta name="viewport" '
            'content="width=device-width, initial-scale=1"><title>Неполные дубли FAQ</title>',
            """<style>
:root{color-scheme:light;--ink:#19334a;--muted:#516175;--line:#d9e2ea;--bg:#f4f7fa;--accent:#067385}
*{box-sizing:border-box}body{max-width:1440px;margin:auto;padding:24px;background:var(--bg);color:var(--ink);
font:15px/1.5 system-ui,sans-serif}h1{font-size:clamp(28px,5vw,44px);margin:0}h2{margin:.6em 0}
nav,.tools{display:flex;flex-wrap:wrap;gap:8px;margin:16px 0}
button,input,select{font:inherit;border:1px solid #aabac8;border-radius:7px;padding:8px 12px;background:white;color:var(--ink)}
button{cursor:pointer}button[aria-pressed=true]{background:var(--accent);color:white;border-color:var(--accent)}
input{flex:1;min-width:18ch}small{color:var(--muted)}
table{width:100%;border-collapse:collapse;background:white}th,td{text-align:left;padding:9px 12px;
vertical-align:top;border-bottom:1px solid var(--line)}th{background:#e4edf4}td:first-child{min-width:16ch}
summary{cursor:pointer;font-weight:650}details p{white-space:pre-wrap;max-height:28em;overflow:auto}section{overflow:auto}
[hidden]{display:none!important}
</style>""",
            '<h1>Неполные дубли FAQ ЕЛЧ</h1>',
            '<p>Кандидаты по cosine E5 для редакторской проверки.</p>' if e5 else
            '<p>Совпадения и покрытие ответов для редакторской проверки.</p>',
            '<nav><button data-tab="candidates" aria-pressed="true">Кандидаты</button>'
            '<button data-tab="differences" aria-pressed="false">Существенные различия</button>'
            '<button data-tab="exact" aria-pressed="false">Полные совпадения</button>'
            '<button data-tab="quality" aria-pressed="false">Проверка порогов</button></nav>',
            '<div class="tools"><input id="search" aria-label="Поиск" placeholder="Поиск по вопросу, ответу, продукту или ID">'
            '<select id="scope" aria-label="Продукт"><option value="">Все продукты</option>'
            '<option value="1">Внутри продукта</option><option value="0">Между продуктами или продукт пуст</option></select>'
            '<select id="queue" aria-label="Очередь"><option value="">Вся очередь</option>'
            '<option>приоритетная вычитка</option><option>сверка контекста</option></select></div>',
            '<p id="shown" aria-live="polite"></p>',
            '<section id="candidates">' + pair_table(candidates) + '</section>',
            '<section id="differences" hidden>' + pair_table(differences) + '</section>',
            '<section id="exact" hidden>' + simple_table(EXACT_HEADER, exact, True) + '</section>',
            '<section id="quality" hidden><h2>Сводка запуска</h2>' + simple_table(('Показатель','Значение'), summaries)
            + '<h2>Вычитанная выборка</h2>' + simple_table(METRIC_HEADER,
                [(*r[:6], *(None if x is None else f'{x:.1%}' for x in r[6:])) for r in metrics]) + '</section>',
            """<script>
const byId = id => document.getElementById(id);
let tab = 'candidates';
function refresh() {
  byId('shown').hidden = tab === 'quality';
  if (tab === 'quality') return;
  const q = byId('search').value.toLocaleLowerCase('ru').trim();
  const scope = byId('scope').value, queue = byId('queue').value;
  let n = 0;
  for (const row of byId(tab).querySelectorAll('tbody tr')) {
    row.hidden = (q && !row.textContent.toLocaleLowerCase('ru').includes(q))
      || (tab !== 'exact' && scope && row.dataset.scope !== scope)
      || (tab === 'candidates' && queue && row.dataset.queue !== queue);
    if (!row.hidden) n++;
  }
  byId('shown').textContent = 'Показано строк: ' + n;
}
for (const button of document.querySelectorAll('[data-tab]')) {
  button.addEventListener('click', () => {
    byId(tab).hidden = true;
    tab = button.dataset.tab;
    byId(tab).hidden = false;
    for (const item of document.querySelectorAll('[data-tab]')) {
      item.setAttribute('aria-pressed', String(item === button));
    }
    byId('search').disabled = tab === 'quality';
    byId('scope').disabled = tab === 'quality' || tab === 'exact';
    byId('queue').disabled = tab !== 'candidates';
    refresh();
  });
}
for (const id of ['search', 'scope', 'queue']) byId(id).addEventListener('input', refresh);
refresh();
</script></html>"""]
    Path(path).write_text(''.join(body), encoding='utf-8')

In [ ]:
"""FAQ duplicate ranking and calibration on cached E5 answer embeddings."""

from collections import Counter
from dataclasses import dataclass
from hashlib import sha256
from pathlib import Path
from tempfile import TemporaryDirectory



E5_SPLITS = ('подбор', 'проверка', 'контрольный аудит')
E5_GRID = (.75, .80, .85, .88, .90, .92, .94, .96, .98)
E5_BASE = 'Cosine E5'
E5_RULES = 'E5 и проверки · все продукты'
E5_INSIDE = 'E5 и проверки · внутри продукта'
E5_COVERAGE = 'E5, проверки и вложенность · все продукты'
E5_COVERAGE_INSIDE = 'E5, проверки и вложенность · внутри продукта'
METHODS = (E5_BASE, E5_RULES, E5_INSIDE, E5_COVERAGE, E5_COVERAGE_INSIDE)
COVERAGE_MARGIN = .02


@dataclass
class E5Run:
    corpus: Corpus
    dense: tuple
    pool: dict
    scores: list
    top_k: int


def audit_card_key(ident, content_hash):
    return sha256(f'{ident}\n{content_hash}'.encode()).hexdigest()


def match_saved_audit(corpus, manifest):
    by_key = {audit_card_key(c.id, fingerprint(c)): i for i, c in enumerate(corpus.cards)}
    rows = {}
    for number, (a, b, split, decision) in enumerate(manifest, 1):
        if a in by_key and b in by_key:
            key = tuple(sorted((by_key[a], by_key[b])))
            rows[key] = dict(number=number, split=split, decision=decision,
                             source='Сохранённый аудит по отпечаткам')
    return rows, len(manifest) - len(rows)


def containment_support(row):
    return row['literal'] or (max(row['cover_a'], row['cover_b']) >= .93
                              and (row['same_question'] or min(row['cover_a'], row['cover_b']) >= .65))


def effective_cosine(row, method):
    cosine = row['embedding']
    if method in (E5_COVERAGE, E5_COVERAGE_INSIDE) and containment_support(row):
        return min(1., cosine + COVERAGE_MARGIN)
    return cosine


def e5_rows(corpus, keys, pool, benchmark, dense):
    for key, feature in corpus.features(keys, dense):
        cards = tuple(corpus.cards[i] for i in key)
        same_product, relation, hard, caution = risks(*cards)
        sources = sorted(pool.get(key, ()))
        audit = benchmark.get(key, {})
        if hard:
            queue = 'существенное различие'
        elif relation != 'same' or caution:
            queue = 'сверка контекста'
        else:
            queue = 'приоритетная вычитка'
        yield dict(feature, cards=cards, flags=(same_product, relation, hard, caution),
                   source_key=key, audit=audit, sources=sources, queue=queue, signals='Cosine E5')


def e5_metrics(rows, cutoff, splits=E5_SPLITS):
    metrics = []
    for split in splits:
        cases = [r for r in rows if r['audit']['split'] == split
                 and r['audit']['decision'] != 'уточнить' and r['embedding'] is not None]
        if not cases:
            continue
        for method in METHODS:
            tp = fp = fn = 0
            for row in cases:
                same, _, hard, _ = row['flags']
                selected = (bool(row['sources']) and effective_cosine(row, method) >= cutoff
                            and (method == E5_BASE or not hard)
                            and (method not in (E5_INSIDE, E5_COVERAGE_INSIDE) or same))
                positive = row['audit']['decision'] in POSITIVE
                tp += int(selected and positive)
                fp += int(selected and not positive)
                fn += int(not selected and positive)
            metrics.append((split, method, len(cases), tp, fp, fn,
                            tp / (tp + fp) if tp + fp else None,
                            tp / (tp + fn) if tp + fn else None))
    return metrics


def calibrate_e5(rows, same_product_only):
    """Select by F1 using only the tuning split, then report the comparison grid."""
    tuning = [r for r in rows if r['audit']['split'] == 'подбор'
              and r['audit']['decision'] != 'уточнить' and r['embedding'] is not None]
    labels = {r['audit']['decision'] in POSITIVE for r in tuning
              if not same_product_only or r['flags'][0]}
    cutoffs = sorted({effective_cosine(r, method) for r in tuning if r['sources']
                      for method in (E5_BASE, E5_COVERAGE)} | {.90, .94, .95, .96, .97, .98, .99})
    grid = [(cut, *stats) for cut in cutoffs
            for stats in e5_metrics(tuning, cut, splits=('подбор',))]
    base = E5_INSIDE if same_product_only else E5_RULES
    coverage = E5_COVERAGE_INSIDE if same_product_only else E5_COVERAGE
    ranked = [(2 * tp / (2 * tp + fp + fn), tp, -fp, variant == base, cut, variant)
              for cut, _, variant, _, tp, fp, fn, _, _ in grid if variant in (base, coverage) and tp]
    if len(labels) == 2 and ranked:
        _, _, _, _, cut, method = max(ranked)
        return cut, method, grid
    return None, base, grid


def prepare_e5(exports, embeddings, snapshot, *, top_k=12, block=96, expected_dimension=1024,
               require_full_coverage=False):
    missing = [str(path) for path in (*exports, embeddings, snapshot) if not Path(path).is_file()]
    if missing:
        raise FileNotFoundError('Для запуска E5 нужны файлы: ' + ', '.join(missing))
    records, _ = read_records(exports)
    corpus = Corpus(records, lexical=False)
    try:
        dense = load_embedding_cache(embeddings, snapshot, corpus, expected_dimension=expected_dimension,
                                     require_full_coverage=require_full_coverage)
    except (ValueError, OSError) as exc:
        raise ValueError(f'Не удалось проверить кэш E5 и снимок строк: {exc}') from exc

    pool = corpus.pool(top_k=top_k, block=block, dense=dense, dense_search_floor=-1.)
    pairs = sorted(pool)
    scores = []
    for start in range(0, len(pairs), 1024):
        batch = pairs[start:start + 1024]
        scores.extend(zip(embedding_cosines(batch, dense), batch))
    scores.sort(key=lambda item: (-item[0], item[1]))
    return E5Run(corpus, dense, pool, scores, top_k)


def product_match(corpus, pair):
    a, b = pair
    return bool(corpus.product[a] and corpus.product[a] == corpus.product[b])


def e5_threshold_counts(prepared):
    scopes = [(score, product_match(prepared.corpus, pair)) for score, pair in prepared.scores]
    return [(cut, sum(score >= cut and same for score, same in scopes),
             sum(score >= cut and not same for score, same in scopes)) for cut in E5_GRID]


def preview_by_band(scores, limit):
    edges = (-1., .80, .85, .90, .95, float('inf'))
    bands = [[item for item in scores if low <= item[0] < high]
             for low, high in zip(edges, edges[1:])]
    bands = [band for band in reversed(bands) if band]
    preview = []
    budget = limit
    for index, band in enumerate(bands):
        remaining = len(bands) - index
        size = min(len(band), (budget + remaining - 1) // remaining)
        if size == 1:
            preview.append(band[len(band) // 2])
        elif size > 1:
            preview.extend(band[i * (len(band) - 1) // (size - 1)] for i in range(size))
        budget -= size
    return preview


def evaluate_e5(prepared, *, benchmark_file=None, saved_audit=(), cutoff=None, same_product_only=False):
    if cutoff is not None and not -1 <= cutoff <= 1:
        raise ValueError('Порог cosine E5 должен быть от −1 до 1.')
    corpus, dense, pool = prepared.corpus, prepared.dense, prepared.pool
    benchmark, missed_audit = match_saved_audit(corpus, saved_audit) if saved_audit else ({}, 0)
    external, stale = load_benchmark(benchmark_file, corpus)
    benchmark.update(external)
    eligible_labels = {key: value for key, value in benchmark.items()
                       if corpus.form[key[0]] != corpus.form[key[1]]
                       and all(index in dense[1] for index in key)}
    audited = list(e5_rows(corpus, eligible_labels, pool, benchmark, dense))
    calibrated, policy, grid = calibrate_e5(audited, same_product_only)
    selected_cutoff = cutoff if cutoff is not None else calibrated
    metrics = e5_metrics(audited, selected_cutoff) if selected_cutoff is not None else []
    return dict(benchmark=benchmark, stale=stale, missed_audit=missed_audit, audited=audited,
                cutoff=selected_cutoff, policy=policy, grid=grid, metrics=metrics)


def export_e5(prepared, output_dir, *, benchmark_file=None, saved_audit=(), cutoff=None,
              same_product_only=False, preview_count=100):
    if preview_count < 1:
        raise ValueError('Размер предварительной выборки должен быть положительным.')
    corpus, dense, pool = prepared.corpus, prepared.dense, prepared.pool
    result = evaluate_e5(prepared, benchmark_file=benchmark_file, saved_audit=saved_audit,
                         cutoff=cutoff, same_product_only=same_product_only)
    benchmark, audited = result['benchmark'], result['audited']
    selected_cutoff, policy = result['cutoff'], result['policy']
    metrics, grid = result['metrics'], result['grid']

    coverage_enabled = policy in (E5_COVERAGE, E5_COVERAGE_INSIDE)
    floor = (selected_cutoff - COVERAGE_MARGIN if coverage_enabled and selected_cutoff is not None
             else selected_cutoff)
    scores = [(score, pair) for score, pair in prepared.scores
              if (not same_product_only or product_match(corpus, pair))
              and (floor is None or score >= floor)]
    if selected_cutoff is None:
        scores = preview_by_band(scores, preview_count)
    rows = list(e5_rows(corpus, dict.fromkeys(pair for _, pair in scores), pool, benchmark, dense))
    if coverage_enabled and selected_cutoff is not None:
        rows = [r for r in rows if effective_cosine(r, policy) >= selected_cutoff]
        for row in rows:
            if row['embedding'] < selected_cutoff:
                row['signals'] = 'Cosine E5 + вложенность'
    for row in rows:
        if row['audit'].get('decision') in ('разные', 'конфликт'):
            row['queue'] = 'существенное различие'
    queue_order = {'приоритетная вычитка': 0, 'сверка контекста': 1, 'существенное различие': 2}
    rows.sort(key=lambda r: (queue_order[r['queue']], -r['embedding'], r['source_key']))
    counts = Counter(r['queue'] for r in rows)
    exact = exact_report(corpus)
    status = ('Просмотр по диапазонам для разметки, порог ещё не выбран' if selected_cutoff is None else
              'Задан вручную' if cutoff is not None else 'Подобран по F1 на части «подбор»')
    summary = [
        ('Метод поиска', 'Cosine E5 по сохранённым векторам ответов'),
        ('Активных карточек в выгрузках', len(corpus.cards)),
        ('Карточек с проверенным E5-кэшем', len(dense[1])),
        ('Карточек без совпавшего вектора', len(corpus.cards) - len(dense[1])),
        ('Лимит соседей на карточку и тип продукта', prepared.top_k),
        ('Пар в пуле соседей E5', len(pool)),
        ('Базовый порог cosine E5', selected_cutoff),
        ('Правило отбора', policy),
        ('Статус порога', status),
        ('Приоритетная вычитка', counts['приоритетная вычитка']),
        ('Сверка контекста', counts['сверка контекста']),
        ('Существенные различия', counts['существенное различие']),
        ('Кандидаты между продуктами', sum(not r['flags'][0] and r['queue'] != 'существенное различие' for r in rows)),
        ('Пар разметки с актуальным текстом', len(audited)),
        ('Пар аудита без совпадения отпечатков', result['missed_audit']),
        ('Устаревших пар разметки', result['stale']),
        ('Покрытие A и B', 'Доля общих взвешенных слов, дополнительный признак для проверки пары.'),
        ('Границы оценки', 'Метрики относятся к размеченной выборке и заданному лимиту соседей.'),
    ]
    output = Path(output_dir)
    output.mkdir(parents=True, exist_ok=True)
    candidates = [r for r in rows if r['queue'] != 'существенное различие']
    differences = [r for r in rows if r['queue'] == 'существенное различие']
    with TemporaryDirectory(dir=output) as temp:
        report, book = Path(temp) / 'faq_e5_report.html', Path(temp) / 'faq_e5_review.xlsx'
        write_html(report, summary, candidates, differences, exact, metrics, e5=True)
        write_workbook(book, summary, candidates, differences, audited, exact, metrics, [], grid,
                       e5=True, pool_counts=e5_threshold_counts(prepared))
        report.replace(output / report.name)
        book.replace(output / book.name)
    return summary


def run_e5(exports, embeddings, snapshot, output_dir, *, benchmark_file=None, top_k=12, block=96,
           cutoff=None, same_product_only=False, preview_count=100, expected_dimension=1024,
           require_full_coverage=False):
    prepared = prepare_e5(exports, embeddings, snapshot, top_k=top_k, block=block,
                          expected_dimension=expected_dimension, require_full_coverage=require_full_coverage)
    return export_e5(prepared, output_dir, benchmark_file=benchmark_file, cutoff=cutoff,
                     same_product_only=same_product_only, preview_count=preview_count)

In [ ]:
import json

# Для каждой пары: два SHA-256 отпечатка карточек, часть выборки и метка.
SAVED_AUDIT = [
    ["6566b8148080fda351b8c945571b920571643b34bfffa826bc7381d36e3bc3f3", "18cc80e9f996b9b231c19d22a9641ce43c1fb6dde5745695c0c3767abe07061b", "проверка", "разные"],
    ["6566b8148080fda351b8c945571b920571643b34bfffa826bc7381d36e3bc3f3", "f4473f73e7cf16f63b3c441f21c00b0af0a4be19d3911ae6cddb3cace7e26bc9", "проверка", "разные"],
    ["6566b8148080fda351b8c945571b920571643b34bfffa826bc7381d36e3bc3f3", "8491f55f498f7b45801b5234fa0adf4b86f5612dff226a35d66b273b070c7045", "проверка", "разные"],
    ["6566b8148080fda351b8c945571b920571643b34bfffa826bc7381d36e3bc3f3", "2f37e4d074ef2e23a7d630a8691e814bcc6f9fa1dc9c9ad7105a1c69e843e582", "проверка", "разные"],
    ["6566b8148080fda351b8c945571b920571643b34bfffa826bc7381d36e3bc3f3", "d037bfc0b142e43c5d71fd8a03ac53e1d5b8615098b2ddf905b428e463278290", "проверка", "разные"],
    ["6566b8148080fda351b8c945571b920571643b34bfffa826bc7381d36e3bc3f3", "73672af682d81f199c72d877a499f8388bdac41e1894257e4197311ab3af0888", "проверка", "разные"],
    ["6566b8148080fda351b8c945571b920571643b34bfffa826bc7381d36e3bc3f3", "bf1fea013476ea7703eead156f2f18fec25742343d5ed3475dc69803267f6b53", "проверка", "разные"],
    ["6566b8148080fda351b8c945571b920571643b34bfffa826bc7381d36e3bc3f3", "7010fc42839f96ca28877ac8335113bdfdc84bb6b1efedfd689a4b26866e80d9", "проверка", "разные"],
    ["6566b8148080fda351b8c945571b920571643b34bfffa826bc7381d36e3bc3f3", "edc066f52fb184f763343ab6272a58b0c271d1cdd717c4e43302f943924bb8d5", "проверка", "разные"],
    ["6566b8148080fda351b8c945571b920571643b34bfffa826bc7381d36e3bc3f3", "34fadd09623b93651caecfd61e44995d25473590d42c44108c0a24d51c5082f6", "проверка", "разные"],
    ["6566b8148080fda351b8c945571b920571643b34bfffa826bc7381d36e3bc3f3", "e1a5c08f5db7efe7dd61d6d12759501669e779fc08c0b0b650ffa819a41c5561", "проверка", "разные"],
    ["eeba290470ca025dca9a6436847f78bb399c422cc1985e762e2fb6ed3b6486d1", "e57c01313fedfbf11a32b4d40fcc1799206ea0ceb095df5d664dc72fa4d63721", "проверка", "разные"],
    ["eeba290470ca025dca9a6436847f78bb399c422cc1985e762e2fb6ed3b6486d1", "d19ae12c45e240cf6d583fa8a3e38bb631c4b237d931c6cb805c8da05d8df0e3", "проверка", "разные"],
    ["57e899e270cf0ff011bf6474922c5f324974a3dd05e9dcbce06bc0a6cf3d90b2", "10bad5002c5cad9ae3aeb10cd61e2202373a63f0b60d8304d3a4f2b795ee83ad", "подбор", "уточнить"],
    ["2d1eb87ccfed6472c1b821e5ddef4e16f12cc337f469c3179d83cafc6c2765e5", "1f209dc610074c6497a40c0378d15e77790decfb7140df12a585d409de641722", "подбор", "разные"],
    ["4311c020d26e345d69b4e620a300ff7443eaa405312b3c8d19484148d03bdf00", "273460c6e976a2dcb194360b87840b557d81643e9bd26d88d1862f80240363b1", "подбор", "уточнить"],
    ["b5e62294cc13ec9560e2015d5711d9d102f6f141622ebcc9e6123eddf56ff257", "371a5a43e047276967e196afe6f552670915b54b7668dbdf4289692d39be5e97", "подбор", "уточнить"],
    ["89043c6f5bff6e844e7d1c3149c29e6751e990ba30982702779509e4e9179004", "cf31459dbb885dd30911e1e174b1ad3640c5f121875ea82721b6e3b2ee4e2db8", "подбор", "уточнить"],
    ["b77d9ab285384fa45829cb8be548560e4ce496697f86cd54ae7f32ac4c820b49", "2df563ac92e46e182f85009655538e6b465aa9558db5cd5fabfd54786912ce0d", "подбор", "разные"],
    ["b77d9ab285384fa45829cb8be548560e4ce496697f86cd54ae7f32ac4c820b49", "2a275181310c218a21fe6ce73e744fe9d758feea25bb7cbe8f99af1fe67465cc", "подбор", "разные"],
    ["b77d9ab285384fa45829cb8be548560e4ce496697f86cd54ae7f32ac4c820b49", "bf2c41531babc5dbc9d0077177277441553c829896772b201448ca2298943fa9", "подбор", "разные"],
    ["b77d9ab285384fa45829cb8be548560e4ce496697f86cd54ae7f32ac4c820b49", "64b33657dc0b41bdae900a18780fe95b0201087bca34cdcf7d87ce4ceae011d5", "подбор", "разные"],
    ["da5ec22dcd389a70fb603f8a701f9095e132df4e0aabf5f6e01db81077eed2da", "b94a6eb463907757009c4ea29a4fc0337866722956e3c1899d473cefe49bcbde", "подбор", "уточнить"],
    ["da5ec22dcd389a70fb603f8a701f9095e132df4e0aabf5f6e01db81077eed2da", "2a275181310c218a21fe6ce73e744fe9d758feea25bb7cbe8f99af1fe67465cc", "подбор", "разные"],
    ["da5ec22dcd389a70fb603f8a701f9095e132df4e0aabf5f6e01db81077eed2da", "bf2c41531babc5dbc9d0077177277441553c829896772b201448ca2298943fa9", "подбор", "разные"],
    ["da5ec22dcd389a70fb603f8a701f9095e132df4e0aabf5f6e01db81077eed2da", "2df563ac92e46e182f85009655538e6b465aa9558db5cd5fabfd54786912ce0d", "подбор", "разные"],
    ["4c673c9a163a72661833f378f199ac420813372a2eaea0b8b9b11e6e7ff4cd8e", "648ed8038443b2b9295fbfa7a8a3bf79f565aeb41085376ca4be1d348f2dff1b", "подбор", "уточнить"],
    ["4c673c9a163a72661833f378f199ac420813372a2eaea0b8b9b11e6e7ff4cd8e", "99b31f34d4658ba3aa10a78b1508eee2b3845657237d0df258a86e625212cb53", "подбор", "уточнить"],
    ["4c673c9a163a72661833f378f199ac420813372a2eaea0b8b9b11e6e7ff4cd8e", "7dc2f4b0ecbf304f13106f86d2ed05f98c7782eb5440b15369c13a4297899a2d", "подбор", "уточнить"],
    ["ac8d3260710b78987b2f2cdc9b5b71eed0c35249a34836a804ed1b2bb70281f7", "780dedbec550b6332c815967c7a062cfdbbf70f7e2c702c486ef6fc7ea0c680f", "подбор", "разные"],
    ["ac8d3260710b78987b2f2cdc9b5b71eed0c35249a34836a804ed1b2bb70281f7", "32aa4063322bd9cc07061087303bda770512761201b8a1e1f507347659191928", "подбор", "разные"],
    ["ac8d3260710b78987b2f2cdc9b5b71eed0c35249a34836a804ed1b2bb70281f7", "85033b420d8ab2e5cf94fc1b829480b09f4ba0495154dd04ffc57a5d3676d4fd", "подбор", "разные"],
    ["18847b85e4d36dd772ab084db044d7966e1eea2eef2315e2b26506d92cb088ba", "0fe25cbe6967beef279b4f9364087cdb3904f31d8223eae1dcc0079d3b4352b7", "проверка", "уточнить"],
    ["18847b85e4d36dd772ab084db044d7966e1eea2eef2315e2b26506d92cb088ba", "896e1c907e218515bdd0ed005e7a9934881316ffab2d7f4b6e345332d4e9bbcc", "проверка", "уточнить"],
    ["7744c19f585ef51f59088d842660a852ff634228f23b45167985aa063c18aa12", "dd09e15ba6912824d93337e3ee7591155dc1252bf65c46f513664c50d1f88bfa", "проверка", "разные"],
    ["7744c19f585ef51f59088d842660a852ff634228f23b45167985aa063c18aa12", "ef6c97429212e051218c7d6b55be58c137c40cfb038b74915eb312e678b01899", "проверка", "разные"],
    ["c4a6834eae8745c98190a6cc0fbe571e8e8d6b70e137e1e7a08353cf0a92a181", "885dab42befffd0b1b5910ba800bf69ff6b51ec23940df8d28d747dfe7383372", "подбор", "разные"],
    ["c4a6834eae8745c98190a6cc0fbe571e8e8d6b70e137e1e7a08353cf0a92a181", "796412fd803c948534a768e1ed4b898ff67f8a04c05bb271fd2b12117d2fa762", "подбор", "разные"],
    ["553e2fbfaf9b1d790a4e17966148c88042b6b12d6b28c8f8bed33ecb7481f452", "425162cf5d55f031339d709f011def227f8647c2fd0ba7f88e45ff918f817f71", "подбор", "разные"],
    ["553e2fbfaf9b1d790a4e17966148c88042b6b12d6b28c8f8bed33ecb7481f452", "aa57bf6ad7c6c21eb64a95cca5260c3aec33983a18415cac45a6c67dcab514b3", "подбор", "разные"],
    ["c4ddbacdcbf2c51aed50ec1b3f10d25d39affcdf77dc6de4e2833285b85f60d2", "a3cc8cf23efa387219e67dd5df28a6a8d493e7c2e6933687407c11413a7dc901", "подбор", "разные"],
    ["c4ddbacdcbf2c51aed50ec1b3f10d25d39affcdf77dc6de4e2833285b85f60d2", "793bb54cc4796647230637995f0ce3c71c74237a4bd011a9898e8d7dae229e0b", "подбор", "разные"],
    ["af5d0fe1d146b70a5646ced956a60e31964e32e9d34175e784f57b18f86d2166", "675b2f0af826b2ea1500af3dc011ba0843b046b2bdd655cd8434cb3779b68677", "подбор", "уточнить"],
    ["af5d0fe1d146b70a5646ced956a60e31964e32e9d34175e784f57b18f86d2166", "b3ab3d3639a958d05f7f9e31cedd1527cceccb0eca3f37d4e68beab2fa8a31d7", "подбор", "уточнить"],
    ["e20bf2f05d51b25a29fd1748c1d7681174b8b030d7ad1f77959b0b2ca8c71ea7", "2de8523927d9533169e9e650ba02ece075b26ea240da47ac0ec23956fdb9a561", "подбор", "разные"],
    ["e20bf2f05d51b25a29fd1748c1d7681174b8b030d7ad1f77959b0b2ca8c71ea7", "591bc9c08b698661f622dd92a40af38b14ca3535f7e6481d9c36cbf0f82393cf", "подбор", "разные"],
    ["740a868812c4bee167ea012527d2064581ed39c5cd2bd8c6cf166ca43614e88f", "d5fd40a695c22ea3570ccd2f5e936df3884c70843b20cbb391b4c0cb13184851", "проверка", "разные"],
    ["740a868812c4bee167ea012527d2064581ed39c5cd2bd8c6cf166ca43614e88f", "413c19a54e874ca0b6f50b692dd3c756f7281080544db35c5a2f1200eb142614", "проверка", "уточнить"],
    ["b8101260aa30c6e1cafefc90b29f72767c1892e959c4c871352588d4c019f1d2", "3672b9ff1222e1b9510ef488f7f8582a4c63cb16c133e07414f20ae28ad3e06c", "подбор", "уточнить"],
    ["b8101260aa30c6e1cafefc90b29f72767c1892e959c4c871352588d4c019f1d2", "ac60c885266d723978659b6b0003b05792d978dfedd4daf244b63a0db828646c", "подбор", "уточнить"],
    ["9d2117a98b48c16f06a0036dd62ea754f749ff0c837d752d3242512640e5e093", "93c5a4e0a94f71125c1b4c52bfcfdec284ca40f2c434abbf2bd6b338bf77ccd6", "подбор", "уточнить"],
    ["9d2117a98b48c16f06a0036dd62ea754f749ff0c837d752d3242512640e5e093", "2a157c935da1bc2b71cc9c0554ae9039492b63037921079365c9c7041edb1a54", "подбор", "уточнить"],
    ["f5ca3d8889b8eddced2f4a76a2f6b9392b924649d1955d36af55c47cc8eedeb6", "86a095d465d457d5191a967e112438d7c6a0066e0ecb7d9994dc0f6e2ffa4cbc", "подбор", "разные"],
    ["f5ca3d8889b8eddced2f4a76a2f6b9392b924649d1955d36af55c47cc8eedeb6", "5ccbb41a3cebae6c611f8cbafdf1961f075c315928d4400586c78a7772f7c9ef", "подбор", "разные"],
    ["bfea9acb53597af79b6614be9d4f08ca94b60038e5ec8c33a239cdbe116c1ad5", "a088d08e1064e07717b6eb22dd1ed30658560dd0ccad97634303719c3412c95e", "подбор", "уточнить"],
    ["bfea9acb53597af79b6614be9d4f08ca94b60038e5ec8c33a239cdbe116c1ad5", "c9a6501e116e76e42660edb4a9cc99418c151be154032b17ad8de36b56b24d9d", "подбор", "уточнить"],
    ["cbd7b9d399b0e757e4fa58df97339a7c27f18a6b47b7dff09a95b3388a3bcf15", "9649e23331f1386f0106a06fed6675652c2cd97dd6eba8beb520f7681ecf9348", "подбор", "уточнить"],
    ["cbd7b9d399b0e757e4fa58df97339a7c27f18a6b47b7dff09a95b3388a3bcf15", "62c611b37f3fbddfb71e8fe7abea30f8ea2c0e4e5af15c72940f8555c8303a3e", "подбор", "уточнить"],
    ["ed7c0c58d817ccfdd822fc88681d8c18d0646af87315cb787b2a8fc820876182", "35a47626f0c78e06bdfdf7a8b3e9a4bbc378e37a7f6ecef38b36841df2b39626", "проверка", "разные"],
    ["ed7c0c58d817ccfdd822fc88681d8c18d0646af87315cb787b2a8fc820876182", "35a264a6f4b504fb348e49a325ca3e5e8552bf5c7336e81fb84c7654bce27a9a", "проверка", "разные"],
    ["7403ae4e5443671a6e71717fb703fb75b206455fdda77afe91ca521ee9cf08e2", "35506b71b0247f514a23eefec1d517142b02d7d07d914f4ba16dc2e1580f7f03", "подбор", "разные"],
    ["7403ae4e5443671a6e71717fb703fb75b206455fdda77afe91ca521ee9cf08e2", "563614f2018fbd4daf433cbf5cb4932f0d5908fa60a21793246329e77519d876", "подбор", "разные"],
    ["43b28403571c4cb5d16b28d09c6c5e07b89cd376ca3725e4808afcae3fe8e815", "94a4d42028d7bd9036896dcc0617a5f07105bb2f4e7a04c744a6659d4a30d96b", "проверка", "уточнить"],
    ["43b28403571c4cb5d16b28d09c6c5e07b89cd376ca3725e4808afcae3fe8e815", "69ed377ede85edf945fa730cffdcf30939bc0dc7a9bf1c9a2baf03292a100236", "проверка", "уточнить"],
    ["cca612a909d53e28000bf2e6d4e27d8c6480d6a66f992097e494b4e4cd18a736", "f7931d42610ed9f6fa969648b4f9a0da408bc237d8f79bd80428b7a8bad48637", "подбор", "уточнить"],
    ["cca612a909d53e28000bf2e6d4e27d8c6480d6a66f992097e494b4e4cd18a736", "f908566656a463f8197cdc2c5af8e2da25c9b2c42ad8c9a00e39ef9362868186", "подбор", "разные"],
    ["58338a07e35c2770e5f874712492ed7fed22d77564f848b6bbde3c329c0de9f6", "77f4b4d9b91bd9a066298486d2576962ddde163fa9590182a610040eef309aa6", "проверка", "уточнить"],
    ["58338a07e35c2770e5f874712492ed7fed22d77564f848b6bbde3c329c0de9f6", "ca483dab30300ac989851dd5e3cb714e8781f1c7aa13c394c08bc97e54d864b6", "проверка", "уточнить"],
    ["88a6d6852e3c548ad5e240c0499232860875ecd0ac45abeb1e864f6b983895e7", "19a94d100080d29e6fe082b1522cb719c5045932462ca9ba3096b355d58b8d44", "проверка", "уточнить"],
    ["88a6d6852e3c548ad5e240c0499232860875ecd0ac45abeb1e864f6b983895e7", "be2942f78498f0e03b4bcdad6d6196a62bbed2f47ee2f9a5f1b9144b5218bd2f", "проверка", "уточнить"],
    ["811173b2be4b4502e08e0e909c4e50a4d42823ee491171cc01dee684c8a90bc3", "6cc423842b9fe083c395eeaf068e79809995721e8cef5b31000a7a200ff4cbcb", "проверка", "уточнить"],
    ["811173b2be4b4502e08e0e909c4e50a4d42823ee491171cc01dee684c8a90bc3", "5d9a49c0382da062de637c590382345f889d6d81942e1a96ce9b2a07161ef36b", "проверка", "уточнить"],
    ["dd92b65d8335e2b9df8747e85f53551b0d50246190e51e2da0d645494d42f443", "49b0ab5f30f5ab0b514edc7af4a0b6617504d8256efa67fb57e605d71c0daad1", "проверка", "разные"],
    ["dd92b65d8335e2b9df8747e85f53551b0d50246190e51e2da0d645494d42f443", "8107af13e3ad1b509c03ecb139d088fc9eb7c10b6dcddf69d2e455a5bed16fdd", "проверка", "разные"],
    ["e0b026a5e9ca3f6abc5d7bad0d7a87329dd2c30f27526b8316d626fb7d4900d1", "8510ef412152b70b931c219e7ab0690f6321abd9ef90e0d8f2084ea0d396586e", "проверка", "разные"],
    ["e0b026a5e9ca3f6abc5d7bad0d7a87329dd2c30f27526b8316d626fb7d4900d1", "948bc7dc115066ad24c438b7c6cf7799631b5dd1aaf8ca565c7db411c2d76d1a", "проверка", "разные"],
    ["896d48055cdbc0096dd2bc1f5aab323b7a45eee2f076a88ee79d8e40fada5d6b", "65e99d80155dbf415a40b667cb7624362e5dd8396402599b03836c81a209b340", "проверка", "уточнить"],
    ["896d48055cdbc0096dd2bc1f5aab323b7a45eee2f076a88ee79d8e40fada5d6b", "fc0c8286d8d5e8def6500b8225ed6815fe857612d24beb0265f02cb1a32a4c50", "проверка", "уточнить"],
    ["ca55e1832248ed7732b36aafaf907062b01caa1f5fc0029f8144298e879af2c9", "eec02fe977b4539585e59a9d5668a8df6fbe5c2c537bde5f3783fbddb3649e02", "подбор", "разные"],
    ["ca55e1832248ed7732b36aafaf907062b01caa1f5fc0029f8144298e879af2c9", "c3bdde274530bea47eb8dae5a4e9963df808d059eb8e4c9cdcdda2600036d7f2", "подбор", "разные"],
    ["c3269ac6a0531e75b0d99a2256f0b66494a38dd920162c8f813f5755c6689a3b", "564f7303237a392edc2223254f33a99293b7dfab8bfe6ea4ed85447879e3d231", "подбор", "разные"],
    ["c3269ac6a0531e75b0d99a2256f0b66494a38dd920162c8f813f5755c6689a3b", "c074020d6940823b38062c6b58a7b3230e8c2af470e6f8b71713ed0b1137ad70", "подбор", "разные"],
    ["e9fa546ae79917a8bd4d6fabdbeea94f13237447a9dfd9fa90beb75999b2cff0", "4707ce6c73c0c4b5e9617a96836ea172655f1ab3452798dd499c9339f9650649", "подбор", "разные"],
    ["e9fa546ae79917a8bd4d6fabdbeea94f13237447a9dfd9fa90beb75999b2cff0", "bf2c41531babc5dbc9d0077177277441553c829896772b201448ca2298943fa9", "подбор", "разные"],
    ["1dc15c992fb352aa6519639b452276aba1a109e9d63b18118ab0925ed3824402", "6888d43deff59b06f283d727864f646eaec2dfb1e9e29a6bc15638aaee94e752", "подбор", "разные"],
    ["fddaa9383a4669e7a7a8ca64933313233e3025aabeb2afa61f5f59c5d4980aa9", "6c324d540fe8586d7f9acb711331d868ad2766513dff42349adb822382afe189", "проверка", "дубль"],
    ["0f8b10b5e722a90b7304dbbaf5442e1e32b52249accad25505392cb6e3ed4d10", "d3b309d8797fc4f5f7cad001ffa9c45e3056ac2f51b358640ebe13b4c953864a", "проверка", "дубль"],
    ["1828495764368d27fbb60bb63c9aaf01afc08719642d54e85da4efa96809a38e", "324bb4de531dc62a1d252f23a2d98fd8d825561695fc3e2bcadd18044824f20c", "подбор", "разные"],
    ["20a6d08346e241c8a780585e915cc1ddd81506e50f47bdc9953c41d02ab9d2f1", "44af43632c5416165bea188b433f893fb36356ce8539bd4e772a9e62efdea03b", "подбор", "разные"],
    ["d71f95330e4a46a24b76888a9a0cacc5862a9aa90795e6de9f1e3cdc1b50af17", "b16a8a363a90c31075d7a044e90592258bfa5991f8164de50a6f17b78f7bc787", "подбор", "разные"],
    ["58246d47faab5322999da931df01709233cfa28478da05a2296584a3bd3434a4", "df9fb744c3702198bbe575a896dedbd4908f3258e382cb84c4bfffe649888660", "подбор", "уточнить"],
    ["764aef954952ddd341f8b6ac7bf4c43792af2c890b6e16411bdd7287e6eeabc8", "7b84baad0dc8798ceb271eb3974b7f753a3264827176d41a44fe48a0a5a00ed1", "проверка", "разные"],
    ["0468d0cd087188e47d46284df9e494f41764052c0234b7ab209b1d0d512b0415", "0d3c46de7dc84e6d4140c8615f2b32fcc63a1b66ccbb08e3394d3e5b5bb70ca8", "проверка", "дубль"],
    ["7dd95e81964731a515bab4d8b20949594722481de7dfdca8fef90a5832cc7d28", "7863dfd365c992096e9502dd30aa3367ac078d6557a060f9edb243fc11571fa6", "проверка", "разные"],
    ["cf28fb3ec3478eb84f9338eac6b766ae33c543b63d0acea0bf461ce7abdc2b8b", "6eed63625edfac01693449492f9b22f3bbb05064a2b04f417312c67352ea63ac", "проверка", "уточнить"],
    ["22b1142d51659793ed2b3997b653cee43e05b1cbe02da1f3f4a8e0c93b996c5b", "f7d88bc7364641a5fd171d5611472eb812cca2e292e1193f8e57f3caa022c958", "подбор", "разные"],
    ["1a69947bc225dd254c74a8b0e2dd39bf99ae91afeca9e1b02ef07af2fee64695", "ba6457170bce3b5e358e25428154ebae22a8b78536b53d9964910f1ad6b167f4", "подбор", "разные"],
    ["7c07ff48c6084d5e3b8647fb35192cd0693f05280371b7795f0bba2ac6c291a0", "5081f4cd2c8627da9dbac51621221cf490235a2b156c711b8ff6aa125356cbba", "подбор", "разные"],
    ["bad7084003e64fdd4b478599cfc384b71dc0e38b99fd83b7679a60c080bfd442", "c43ff42440a29b34a65e7d85f20837679f0f92b592ac3914f683142ca7528e77", "подбор", "разные"],
    ["77662cac04a4150fb6f22dbca3b34a5f4ab2a369e0df4516e276130cd37d9e6b", "32f04927803b613cb740083e9d3d344e98710216ecda4f0c8a6589a636ae6955", "подбор", "дубль"],
    ["52b2a32c3c5192fd9d15d4de57b2f4c83c2aa30122da3d616cb1d2abfa47178e", "ad9e86598afd6b0417ba6e2a24b163a64d7da29736b31640711257b45cfc6b8a", "подбор", "уточнить"],
    ["2a275181310c218a21fe6ce73e744fe9d758feea25bb7cbe8f99af1fe67465cc", "bf2c41531babc5dbc9d0077177277441553c829896772b201448ca2298943fa9", "подбор", "разные"],
    ["abc574dea4d17b7194a30bb91d5c7e5910fdd9ccdde10f7cf317a4b92955db79", "b8546cacf94a4d5d0df58bd358d81f3a722ea310c84f100e8785f31d21ce9c7e", "подбор", "разные"],
    ["e08de39aeb4e393c54099638e1fcd0e3601cc8faedd5e53b0e52d1b706363ed3", "a513ad02506916ef53cf7e02d3b1d39ee6f98f455ae418de58b7af9d8d27d04d", "подбор", "разные"],
    ["bd0f04c6f831ab0c832517d9b3ad0b98b72acf978579c2831ae69cdd5e2391fd", "000d8ed6bc9d513cf1b4f254242149804a60bc37900e674299c27905f528217f", "подбор", "разные"],
    ["f20075cebd66a2699c8829920b4f75bec50e522bc1226963e98073033b90893f", "df74a156fb3237e5e2722dd861c88b8ad5ca2562f0c1fd095eeb4ae919c1347b", "подбор", "разные"],
    ["168ec9060f6e94ea6403b1d6135649c26402af6a983acbda10b9d329834f8d42", "2f2bdff4649a4eae4fcb0cf450824bb96da79245b7c4fba69e6cb3095ba32567", "подбор", "уточнить"],
    ["7c5dd42d5eaf3220d48a3ebfc796c591f8f4254bb8fdd9321bfd32de403a01a5", "fb8e1782636e76630cf89aaa4edcb0d1f72cd63b13dca105ccbf21c8ef55a7d6", "проверка", "разные"],
    ["caecf23f6ac6d191f121d12235ee57ba7d45d49324f99db2269666dffb77d966", "5ddd71b0e7dfc7376c587812e670e609425ff5f522c6ce32d2451191a849c2b4", "подбор", "уточнить"],
    ["576ab13be1c11702f1eedb263e726cab1d3aa5badeb6f91fa07c6ccc4cce2af2", "b045b43f6b5b586ef655b699a02863b767712de2eb904ca62f3248ba0f841e92", "подбор", "уточнить"],
    ["48911173e8a9f32cdc4dc696483bc4fb063a7850852898d9dc3325ccd5b7f19d", "b73bb64f4fb2bca3006ed182719f180856104d163d19ee1a93b1f9a25c6fc88c", "подбор", "покрытие"],
    ["7457c60847ff0fdfcc6ba8acbdf1795df3e29b9711a578b7c65cbc21325495c8", "3d6eb8509d9bc0188693739a6c7b02ce0734538edd3e1126ee1f696b9e5b5b40", "подбор", "уточнить"],
    ["20f0ffeed4251f7807cc14ad1b7992a4a819cfb42f26bdd3ee1a0c1ff8def44f", "136566f0140eaa0db8f1dd547608322be75e707a7f9cc8ca1892032c602132cd", "подбор", "покрытие"],
    ["bb586dd1b82918441731774211647a9c2f4be37895c190b2db39e99b790d2d5b", "9aa4707a8c29a4f1a4438ea710439256305279e84af46b0faa9d7f6557e381df", "подбор", "разные"],
    ["6d477320ca664a21c1254416413a4f96b11cc2d11237cff211080f260f0001db", "52e8ca71fcf07f2b6456c2f1a59cc22449d1aebec2bd27b58200effd24e03c49", "проверка", "уточнить"],
    ["c0329ca5875a93a37108cb63251bce7550f69da5f6935928f475971ec9000afc", "6b459091fd815c94f727a5882307c93adc491d83ac39da8ae2418520f75adc16", "подбор", "разные"],
    ["62209bd74b44795cdca5ca693a7d07cf7e4ff3178549035ca0470c82d40d0715", "175e8302b7ecd08fa93bf8e6ddd935c6a070106cd0f66a4cb617790cba73578f", "подбор", "уточнить"],
    ["a89a97741d7b04a21e622f85a1a04a9b81bca0b32579f4508b498464e5e43ba5", "18359f5d90b9fa57ec4ae0dd02de08b852fc65ddb46ddf8edd9b9c825157b550", "подбор", "конфликт"],
    ["78d92c28e3afe96a8f78848b288aa5ee2111a156f5ac457ba3f0d427307d6f7a", "5ee4a2254a4c20b68b4d1223e9030653ddceaf6679cfef9d6a16517fd11d0624", "подбор", "разные"],
    ["de4bbde78e0dc63b6d7bd7ed50b8595b5fd5634027c45087ebf1c33822479575", "bf2f4130814138f3e2c06e1b56727ddf7a183cbcd78773b9a3a56ce0143d84b2", "подбор", "покрытие"],
    ["5effcd381d2d7b25c0339be0030f8ca1f31a33f51a52669687c72ab4a2e6ae95", "673497ae6a372742a62df92f6c4a209d8688382d390ca6425073ba1f5bfd608c", "подбор", "уточнить"],
    ["22a7c25de2ca9f134e39040e95c19dbacdcba28104a0ca08325aa03e31e8871f", "e0fdee62927ade92367592bbe8149ed875b23733fe2bb3cc5b1ad50ae7ab591f", "подбор", "разные"],
    ["becc490a4cdf5a91b13e2a34dfd445ab3d5e8549edecbe73721e65ffa02397bc", "bf049e7c2d91c6da12a0b60371245f6ee42798f6ac69976518a204da9130502b", "подбор", "уточнить"],
    ["3e23669383e4df63e0d771168f4d1c45fd1f366d76e7ebcf8d1438bbb00de0e4", "6aa5e364fb4b303567da3c06d7d46876105051e86ee83681a7165d3b3b36c254", "проверка", "уточнить"],
    ["5e6c80ecb76450290f06d6a7157078ca0902450f6aa71df05d5fcd63e2429cee", "07f3849e5182fc0d181577b14821bab6bb647d128aaaaaeb547015311e11ccf9", "подбор", "разные"],
    ["9cbd92e8e35ccf9e6f75ad535e42c40237fbb9916c46b0ad73882b5cf7b888eb", "6b2886ce847b06f9c557057eb7450c759d0f9e0df6b43ea5d58dfc27db6e3743", "подбор", "разные"],
    ["32aa4063322bd9cc07061087303bda770512761201b8a1e1f507347659191928", "85033b420d8ab2e5cf94fc1b829480b09f4ba0495154dd04ffc57a5d3676d4fd", "подбор", "разные"],
    ["bd0286ee53b406b2d118b37ac752e4053d8423649e4b6a04f30bd565d4b2b229", "2379d1e283ed1a3e5beaf9d1591b81102946f88a468814fc1090bb2427ba0f43", "подбор", "уточнить"],
    ["3c6f905f14e3c54ccdb951d44e504e5ff4e8da8429f1445246e41141efa0ba68", "edc4b5c8706856a5e9bf398d5b5615553eeba08c98ad3ed5cfc889c67de2c7b3", "проверка", "разные"],
    ["2ab718ecd0be0f956123a68d3af5944ea7bb9aa54f4e181856d3980817a5c852", "b26f3572beaad337acca77b966099efe4ae1cd19f280fffac32e4b711427d364", "проверка", "разные"],
    ["1ba7d409c42e46bdb9a1ecfc96ae7349c7aacb095d1bff381496525eb16d6eb6", "f3163a911065bd98e03cd0febfa9ce620875d1118980b9ffe86bc78232c68932", "проверка", "разные"],
    ["cad4739727c9720705d3f514c5b526316b90f260589a6b1ac696c0df91692c6b", "7d940ee53acb6d4507b8512b4ba1eedec76e9e9f8a087b742b89f5efa68b285f", "подбор", "покрытие"],
    ["a823aa7e115f57c56cf3076c44801c9b1aad2bc0028db32db27863e3c4b11874", "056a1c6951e9642074eaa265bba1ba7f971a189b040af8df831c400fbec1637f", "подбор", "уточнить"],
    ["53ce280102f45393f69d36e5e559a567ccfc31d96ba453517cf980ba6c408a84", "e18702f23c4f2d3af6ff073c6f4247570c168906b32230d5dba9835ace100182", "проверка", "уточнить"],
    ["6d982de8f04c31d6b3746e7455d689d74440b0a55dbefd836a9f827393b9bdf7", "2fc99f4a6eef81ebc4a033327e3bb4aac7d7ee414f8a3b966f428a8574e78ad6", "проверка", "разные"],
    ["a72f0e2db60868ad1eb1b740de40afa1975cbf934a9745d3078e7fe15b9b2a40", "90291d0e01cc510ed16ffb0c019936aed5194399842991ca8b1389e796a628a4", "проверка", "уточнить"],
    ["e749d0fbeaeea1b5723ef983af0d3b8563527f738a3020f47ded4807181bd201", "21250369cb41cc487bdd5580ee5cfec05806f6225e795446f15f6e0070bfd2b9", "подбор", "разные"],
    ["511750d94007dbf9b4ce4518108df38a7b62981ce8dd4ef789c494edc7aeea06", "208cee3db311cacfcd523bd400dfc10a4bdec952772e363a1ce43c220a1f4ebe", "проверка", "уточнить"],
    ["a028068ce6909ae7616bc102c5aa3e17b35ebabfc1a93883f9aa6d21a69a4039", "cb7f8a6f84367e1897687e555da39d96ebcbeab05cd5332a6b229b533905b90e", "проверка", "разные"],
    ["cbcfa087fdb279fc5e8ffe154074f0dd8a86674e24e3f488df88c504173ba8e7", "e5a0df3615e4160768f75a356fa70d1b95deac9200b8565ad1ada01689291a6d", "проверка", "дубль"],
    ["75b4e15a553dea3642b01c77d3ef4e73c8857f3590aa9a6c8093504cb32959ea", "1e133427609a8e6813499a59313e5bc18faae337c32aafe7ebbd1b3172a41db6", "подбор", "разные"],
    ["7d96662b9b35bbd26eb7277903e7a93f667390528ef059fc9deaaef86eed6dab", "c62149535500b6922c078d1c31bee379528216a3ec9f1118feb99c41a4a07372", "проверка", "уточнить"],
    ["51adc792a3e21339585e4acfdad0dcc33e7489dab461829d379f371b6f37b5f7", "27d6d42713df1c6fd230bfd55fd993c15d8a3edb58ec972e80f6ec0bce836ece", "подбор", "уточнить"],
    ["90f9de8790ca223d154f212aa5a4be6b253f8a79e4ea13c87ce06ded2a30aa7c", "684c6383d9927c36ce534eed6c4a3588509dc9ebcd836390a2dbac74412c63e5", "подбор", "уточнить"],
    ["8374a141a8d27ca415ca2c63d7d1d0438fc5b54d4e5c6f479d2cc57963f62977", "8fc4e8a97fc746982d3f31dfea42f33b2ebb69fd8e92a0b0d5606c0cf013b1a1", "проверка", "разные"],
    ["a3093ea2087cf36e6e1408129f655189cce38303b294b5184f6cfbfc28d5ae13", "dd2c963ad5851de773d3219d5c8d272057b6368bf60da03d2593655481df5344", "подбор", "разные"],
    ["11a7dad720127d28a5b738eda46bee950c67213c4d44671efa6cd09fd85aef90", "18f563c61f1c7fe4046ae45ce623add60ec4132458db723e15ccf64cef002fec", "подбор", "разные"],
    ["cf1898555c055ec2d4d5855228e4b6367888e81b1cdb07bbd5f57e9182d05429", "c20846989b77350ad81fbd51fb71ff872a7942cb0e938fcb00c5196426e1e53e", "подбор", "разные"],
    ["7e3b39abb971e33d8c875fa1d9b277438950f689956676bfea488bc65b4d017c", "27547cc4db1a29d64e5dee0e3fc7ac7722a1ce90a3c2ea5334df23cd5cb2e393", "подбор", "уточнить"],
    ["18d63af46f13d6aec16ee49573d84eb2af18769efb791d6e6807ac259778c297", "fb8cf2fb033d15086497a46a6390fd9213fb86f81d9140a995173120c6151cab", "подбор", "разные"],
    ["519bdc8ad59ae6d15446597bfdad9de00bca3d0cad6cb0703324a2c273b71a6e", "6b5b3d81c3631208aba9dfd085f0223580f3b64f9f11b752ee39a03d44c89b53", "проверка", "разные"],
    ["cb6899569831638fbe52b4acc6db124e810d8a650769e981e894a4fff9d1f39c", "a0ae6b7dc2f300bfc6ae969ba595d6d7cc676d47014c0b78b48189b6bdb2ddb7", "подбор", "уточнить"],
    ["21cdd69832cdbe99501cad4d02e12cd9bc1d658441cdb949d69787894260e5e6", "82bae3ac96b10de3adce7044c19177c63dd2ed1258aefbb0b55c9feedd09be65", "подбор", "уточнить"],
    ["0b1bb21a964c8e0dfab290ed780f8929d66070f96c63eb0f8268ea9d65ac55ee", "524c6df956b18f35991dfbec14494d9d72217b7315e82b794e278e77093db0bd", "подбор", "уточнить"],
    ["7ea84effe5fbe411307f3995302b92bea1f3220514b586d3681ca2b5b6cc4e71", "bcc981a540f15042358d498f2c591c25580b08bc073d7391689a2647c90cdf90", "подбор", "уточнить"],
    ["196016793ab2620632aa07961fec0d3a4713660d6923a32903ae4ebf86c09da6", "aae320f64107779cd7b94ab1ef1a565f3efba5079bcc616a88d7a7bb9e8e590a", "подбор", "уточнить"],
    ["33368dc53985845766d731476dec196904a5a1c9836d82ba684f282d7f593447", "5e40c9a25b7a7d1b82f6de68d486298ba48c724f84720e13ea58c476f18673ff", "подбор", "разные"],
    ["ce96c7af3a248922aca877aa30387cd5712c3d706c7a16d2f61d1d42487f320d", "adcd1b2594dde107e7b8487cc9eb58fd0719bdd2f4a48c9e9323ea8dbdd9acb8", "проверка", "разные"],
    ["7c203a07a5d314aabd0946f62ab62ae048c8b86aff53a6e4b9023b5fca248dd1", "a5a584283d93e54c4820d5a9f3ad7847952c4a205071aa3367c1c869ab6ab905", "подбор", "разные"],
    ["786bc03314a0b487251315173ecee873f05a10098031f8247df52d709a1d7f93", "13d7941172f7d2471828f6897b38d275755a40bc699b1d617b57fd6c6423deb8", "подбор", "уточнить"],
    ["33bd8e8864a5f2b5a3df5c211c832d122534dbeef5d21537eb35382badae7502", "7691121c044cf724f1f205873edb1903e9c4067dc18c124b487e8e38cca099e0", "проверка", "разные"],
    ["bf7cda23071f5043a72dddead3f6ec62489f91c325927be8707867483885aa57", "719235b5d25e402ff9721a2c4408e8595cb1828e9529859c0bd65924f145d5b2", "подбор", "разные"],
    ["d46076806545bfa65adbe69e7ef0a40a58bf30e1bae9207552c055de94502809", "ee49827ba5ad115b0ed3aca4f0f2c9ae7869319929f38fd4952af23545720860", "подбор", "уточнить"],
    ["6988574086a9e25dbe310df598ed75fe7f3f68e21ba42f5c025df21c8e7b1faf", "1a59a09a6efa310a5c6c989490d1fcc384770460d6b97c641c9ccf332499b743", "проверка", "разные"],
    ["c788c97faa8bb54bed75c070394b63d0ac50c27717dcd3691e00be1a6f76b283", "237882084c4e0f21885a92956a2b4e1e3f61e0a102077bcad531aebb1c81ba72", "подбор", "уточнить"],
    ["73b559917f82d1080f4413f00014b1f5bc6271d4e817227bc6ef5466b393c5f4", "0c01bc5ffe26f34e5211983382f71360eacdf8099a563386bc08c3e7a93f96db", "проверка", "разные"],
    ["bfce801147a52c77f9c075b7a0d3f0af4693b87fdd7b54825e045139b84b705d", "0e678cce2717c1ceadb490a0f3a60da8399d49ad5302a457bad94146a640f871", "проверка", "покрытие"],
    ["53ef898f79b4a3bb6b1c71c982e021e96d354c9e2e21a3e3a54111199dc03113", "cdf9c9a5e8382d1cbd0634d4216524f6f6b5ca056bb828c21ad93f138c3d4230", "проверка", "разные"],
    ["d0015445de5da571ef59f00187145cd41765bfc8cb6887dcc1be9a32a991ce69", "b3ccee8128a772aae2c53839ac2093821e9023d77070c861e8fc936678fb8601", "подбор", "уточнить"],
    ["9dc57fa99c700cfefdfa071c9c67548f2fc87df9dfcd53413e6402b1a02e859c", "ab1836147d3a13147b084642413ef3e40fd18aa9c93c74fcc47ff11584f4ca7d", "проверка", "разные"],
    ["42adb2c20aad15264af891bc9cb0f3974c48dda3366dad11c44a2f1b3220f51b", "bc4bc5f91e5591e671a6eed7ec3ef29b8a5c1b1219084b7ef82318c4297a657f", "проверка", "дубль"],
    ["3cb65947d901c700a10fca9f97a75e7363ffd92a4831df7d614efc749a59e5b4", "a991853887b544f01ab1c39a1ad51049ca803c1f049b43e52818e8d439fc156d", "подбор", "покрытие"],
    ["e9009f9153b9bc46e114f4aa40a1f9c3b0bb962de275dbd70c54f768e7cecaa2", "6d17e93251e8691fc6822ef1e398f22dda127fa6f190202f62a4298faaa9dc70", "подбор", "дубль"],
    ["d5577a18daf965b8d4b69dc1eb20000a5da52b778fa2c154a022970ea79876d0", "09ba644fbd81e2b95f9e345a3ec4440df96c3fdf95bff28c3e33a5170e4eec74", "подбор", "разные"],
    ["05e42519beb8d575677fe8eac1df2850b443f3abb38e29b24d3dbc81cba83d47", "23e34c41153ddbaa349b870ea79fc123fc4d9bdaf1fb8170b9a681dc586f5d62", "подбор", "уточнить"],
    ["ded86918611ae1a6fb325eff9f9bbdb389bc4eae5f7d9462762343de77125755", "8536793745dd623fa9b8e1a62ecc92811bb6999b697605c5da4accd594abce5f", "подбор", "разные"],
    ["6df414396265902502a9e0fed5ef427ce6f591a39e2839966587254af22b2b38", "b3a44526684b59f133e8c8ff6cd1e4b23e2fe97bd272ed9e6170199c4c97dcb4", "подбор", "дубль"],
    ["a33e1aa36b5835de661e533c0259d907619c8167c27571bfa92d130bf67fc2f9", "1db2b4038dfa53d93eb237881d22ac87654ae69275dc843e93892b691554a2f7", "проверка", "разные"],
    ["e0fe9524a222b3aee0a8733a5bd40425fae9237badb06e649e3a3ccace7cbd82", "1d8a2dafb30e08fd5c6bc22a9281629318700b1eecaf39e291e73219d40411b4", "подбор", "разные"],
    ["41541c94ffe0f5a78aa16f7c2cf3b8790585d936a4d7c72ce88673b7b1c44637", "e0a9928549b2a3aba003ef9f3fadf0f586639e1c3281f1929b9672edcc271c72", "подбор", "разные"],
    ["568d75547e181f6d875cad9e66161e07b92e7c745efd3a3e1b8fa48744708c90", "34487b56f5b73e86de1f084c782b21005a16059f4dd7adc39eb62ef56dab4ff5", "проверка", "уточнить"],
    ["2f659471871d852918237d74d5c2af9b8bb0f8c1e2ee4e7e04daa2357aaf9467", "241597a3a4f38a9ee18be5c3e7b3d055ec0181c8c3bcb8f78894ce969e7d319e", "подбор", "разные"],
    ["3d96dea6644e7ad7dd5f400b57468946bd8ee1debd18ef0b82fed06145a99687", "52689ebf4d1e197ef22da5dae9029e1d929d412eaaaaa37e24e8f19360847754", "проверка", "уточнить"],
    ["87d3ed8f57e8a3742b963f709b7253e9cd4b3469bb1ef99a597c1734b5a20ecc", "c2c5c1992a2fba58a2f6decbfc900a862c8530295c38e6e0a2b5e0503a9273b1", "подбор", "разные"],
    ["2b3c11dc8663399b29867fa632e369eedc09f9564da8353124719ca7b849723b", "2e6665904a4826da583382d79e4bffeb3037349f149ec63034c3e7447fd74285", "подбор", "покрытие"],
    ["9b46d23accc0adea4209ee2c953a619e71cb2e7e4a4d6eda51de664055275597", "2d391cd394e701f6c09da62fda3f86fd5df9b391df1924b1b1f95454f04642dd", "подбор", "разные"],
    ["74ca9dc9e23bcb9e93c3da44f77ac30c4ee097056c9406c548475f6946866d48", "0a7b300eb8a1d3840fa893c8ad027aca3619943cb5ddefd52a89830c24133dee", "подбор", "уточнить"],
    ["1b9a7f52c55173ec85a79c1684087fd838b4d59e168728e812264628193a7875", "882bf52565608c332a6ca2c7c3a15dc4fbf7d73b22bc9857e1dc4482a4dec47d", "проверка", "разные"],
    ["a98787a26e8e2bc62f401ca763310cda6be78e3be2540d21673824b40ed7a392", "d69c5574f1d4a51992c5677039f1f3c2a657f5997f9d1ea4ed3e8a11dbe91b89", "подбор", "уточнить"],
    ["80f620269fa57e59d7a2fd053e991703c20c59a5490b6cf6955f2e2bf1b925d8", "73cbca8db297a1faf8f3e5b5881c5df86a53f7a23f3da04aa4fb7bc401ba8fd1", "подбор", "разные"],
    ["52bd1dbf67c1e4ff22b366e254674e8155fb51a7943fbb143edde88720f6912f", "85b7c1b114e71320614be58bdfce37499a2ed1a4e29adeeec4a46b727bfa3a54", "подбор", "разные"],
    ["1ba105480147547245b1b7ed8a0c20fd37f8bbb2224c0480b581e944a30a054a", "bbcdbee918ef3cd5b84f6249f5d5f588824a1ac27a6eae825afd0843324186fb", "подбор", "уточнить"],
    ["42253d00ca8c6c4ba833b1999da1e15590c34714a0f38ccd84d10add98403f5a", "2fe163535729e59c2d2dfd5921035d501d1c4ffe46224bb39c8d57c975205b1c", "подбор", "разные"],
    ["48d6f63e6d9ae4576f3521d10f3c8f0a1c4ec5322fd18096b58799c11da0b61b", "b6908fe838bddc47fd63d3f47e4d76280c03d4d63f28d71180d0f5eed8426c25", "подбор", "покрытие"],
    ["12d14e376d7841782db51849f5b28dee4317f1bfa3c85a305442d0978d735bb2", "c0a1cad6b266809ca600aed631b795a9f9e2b0ec2193277b875ce48f66155f03", "подбор", "дубль"],
    ["d4b36158ffb7d24ca1cc5e8eb6ba5849228eb0a7626c37682e0c06272374d978", "7736685dfa5628aa67117449ecf5f91d823d4bc18256de7f303d060b975d4511", "подбор", "уточнить"],
    ["f88c47e85672acf280d0ee6f56e45360d01c4c3eb740f5bde957b519b276f357", "2d9c4649a03511f4848563ca063ae2b6dac7772c74db39d366904345c94a017f", "проверка", "уточнить"],
    ["33b09f11f86eb0ee8f8e284ebfb86432dac0e04977efe3bdcff76342c5719073", "cd4699e3c2f61a87c427abb53f8abc3056137ad128be8b9bb9e5e2d9701f77c8", "проверка", "разные"],
    ["ac147b5608fed8177ed100fed2bf30ad35461bf9b2619dd26df32c08de0878e9", "273d7eaa5bd578ccee8411bf57b18cf4eddd018e0e9a2807d87bf169c08fc6f5", "подбор", "разные"],
    ["c94b7e4c29416cee86f881466ba29d6c03db22318dfeebcab8504718bf708b02", "96a4650cb1645dc859ef8bb50ee0d1a5fc916629ec01116706b8720462ab1105", "подбор", "разные"],
    ["db0a5c817ccc88c1fc71c64af68d1350b31bfc6882cd65aeb1b243b644a6cb28", "ff79cf1118d9d60c57f0c57a5fe53b904ec72c50b0fb2f682e24c5a0cc31ea3e", "подбор", "разные"],
    ["8cd361300185686f33bac7dcc8b963325bbb950366be0eb7e08b6c48f46e9371", "9eb063b779f1e9c36096d654b7cde01b950bae9ee94dda98d7b43b263fba5e41", "подбор", "разные"],
    ["4ca19b4b54d4da47a300f077872de924902fa557834d54676a19b41264e234cc", "3bb3ebd59e22891e983fb7ba9bcafe18efe793f26954bea96d0f824bd374632d", "подбор", "разные"],
    ["42b0fa0dbde1dcf190c9093627467cdf8ec4f127773e77d915e72dde4545c8e7", "b14ae47c8568f92a1c6fd5f7a65ffea2296ec330c62c304c1040ef6f07517376", "проверка", "уточнить"],
    ["831d9243c22231f47102dcbdc376b624198e418ad9582ce7329425cef132ae14", "e14dc5ededc58f3664131b08c8c8f2516b84cb81896eb46d7a56051b6ec233ae", "подбор", "разные"],
    ["898842e36b211646a661f2c3b15aa47e061fc2d6af920b6701a3232f5fb06da5", "c17793a631fc30acb728b91e5ec0a178511376c133f8471a2f5c584b2b5e1312", "подбор", "разные"],
    ["2937c5bc76c7e69a9ce04f7e8d27c9b82b39c0984fbbd7a00f413209c4f9d73d", "cc23dcc15c9a8254b33d9c3653c10a2fb4ec2104901ed67b4eb5195f818940f6", "подбор", "разные"],
    ["2bdd38426fd1f55dc66fc92307cbcfc81e68ae1978cdacd64c13acda9e2fca2e", "6bfb79d3e0d4bda5dc193beadcfaa077a49d55b31417190f6055042134cd0e7c", "подбор", "покрытие"],
    ["edca688c8a4b87ddc7d2e6eb452106353dd5d4be138fc3bf0f4d6feb87919b41", "c2f47c1c3764a158479d063e74891c725c2c17d53e31913ae6a6655204a59c87", "подбор", "разные"],
    ["0f78c8066d41ffdf471cefac8811a1323de1b763ed92d1033f9701a8403e66c0", "211617b21704819643b1629c8eb2ca45a4458e44ea6d39e0a7de320bddf45ae8", "подбор", "уточнить"],
    ["0c403829e6c9cf157da0a0fa45868899bad478f63871e64c6cca2134448627fe", "cec3ed60748bf94466ef96f1a3843d6ed0c11b1011069003f27527863e719898", "проверка", "уточнить"],
    ["8623c0094af129336cd606c4466fc6c865ee316e999eb09455c471a2b27cf18a", "431d27690cd1002af180b0c004909f690ace681279c5e82f2f7f5278b0e433df", "проверка", "покрытие"],
    ["aa6b3c6487853ee559aecd30bc0df6048a388cbdf29459af36fc4cee6356bbc5", "eec9f338e0d7e586347f37fcc9ea4bc9b90a31ce805727e6b4fd66a55be8ee4d", "проверка", "конфликт"],
    ["eb85f2c2d42b71673a36862fe1cb63c4fc57d5e35297726a89512ddaa6cb6e10", "69c94725f1f4b8f219bcc315de35e567955af1e8bb0ab587a52b67cf353fa0ce", "проверка", "разные"],
    ["114c77cbb7a623ef7e6e1dcbdd9da4d65ec171ff22a8546ca4ebbc4c4525bb8d", "35987c21b9f7c6e5a836a05772d1a238a457fcb134c1e04ee60d7cd86f6ccaf3", "подбор", "разные"],
    ["595192e54987a3492b81587761da30eb1159b31f8745703ea2833994c0877918", "b54c87f0b800b48627a0c1b04e8eaff0ffd0a8c9a9844134c20cfee87c03bc45", "проверка", "разные"],
    ["5d32996795abafe4186c8b9517c7f4752e7dbddb078c8edf6b1bce95149e24d6", "150ab6f144707aad278c1a9ec6c70f608a84c6b64e1efe4aad73a47d9b69b374", "проверка", "разные"],
    ["55ed7746d15f41df12a7cee2c13fe4a8842db8e6d4ef75455ea4ee42023d0aa8", "68e595051e3c84b580994c1d509f8269ce7a31739d8f1aaf8ceafff83476ae3c", "проверка", "разные"],
    ["a51f0ce95f8776f42acad96f6e1a2e1f7157e1d4aba38f370a6b1846b6549ca6", "63ae7d39e7462763b72180e6d27a02305e247b7b24e84043e466f00d802a5d11", "проверка", "дубль"],
    ["8408537270ab644c5aba7d3c19fc8fe98936a972cc3679cb80209cd53473b70e", "8e1f2d2d59bc52263211c00e1745c7e8dfa8912900acd679a171ec04d54e0b9d", "проверка", "разные"],
    ["43c49bfb96d556460df6883e809a25e230bfd0910ca9a397dffe2f8836fdf959", "325effd0ddafd4d1e724f07217eb5f8e3c5ac5c09316fc3bc1f4012a43bfbfb7", "подбор", "уточнить"],
    ["2130a4b36c6a1bc3195a2a26a3f4648d84f396de3eb1bf8c41dfd2f8d362a29d", "c3a7d8eb48bc5479e4d8b6b6e5272e01fb641972f77ee1dc74b091d61ae30b18", "проверка", "разные"],
    ["aea6241cad11a42f4d13b1d5d963bd11a3af6313ad5753a502b9b55b1e1bd643", "cc34a726365b175fe610cd3605064d970b06536a37f9140690e68c0790b3e073", "подбор", "уточнить"],
    ["61c20f84c9533aff899a63bf9b7c93e3ceed9dd924a1640d4e19f9ae13a5fb67", "eb87eb215df44cae6e7001f303ba83c56b7812b6153fe41128b8210a41191d07", "подбор", "разные"],
    ["e37d0f98b3f0f543023fd9b8a88c455355d44ff7da55bbb0fc7fad0febb81273", "9c7caa58a431697c646bea8e458103ed80a1bb4b8f20cf7986d7db8c655648fd", "проверка", "разные"],
    ["4eb81b26a6774c0df8a27403a47e0f7d70035c6329e95473e3de6cb52d28aee4", "1065ee861c5e864d987b6e590d0839b7021f3e7deea38d06b2f7b6de14d2613d", "подбор", "покрытие"],
    ["a3ca14ea35190ab3105b820ef2e6dbe1003033edb02a235eb3ef64089aa90d6d", "8a82585bcb13c90cf3a0368fc219e1a8ad45bb35c860a88fa9bab9b09d9ff953", "подбор", "разные"],
    ["2ccdd53cd2da002210a298b75dab07aa1b4d736279c166088165ba74acd682a8", "824e3cf324f1784f45af6660896dffb7a9419ea0a2548f94d85ee97f74fc9ea0", "подбор", "уточнить"],
    ["4ac88da67bbef408ec6e01d3ad6e6d4fc409b17d9325a0b9d3a4ba6a8e98c383", "2596ac0dda534eae14985db8e19aa8903518afad0ceb6b11a3c0dcbcacbd3eed", "подбор", "уточнить"],
    ["4c804cd0f5645b84e6ea944219f18c4abe4a58427a33ccaa49d6f30d0ea61888", "48be2ebf6eb5505861f8516ad12ad35524cd26e992a967f21787cc7282209ea1", "подбор", "покрытие"],
    ["183dd860c8db2db42f284d0335e9ad8e503c4319a1258dfec7a8c0c72c7783b9", "418d4b4c82da7d6cc4fed2c033075bb9c7931e597fcd8881fefa4f36829de163", "подбор", "покрытие"],
    ["a8fd909e906030423c18ece755e588affbd9ad8db04dc854cc6686339718b9b0", "5833b2ce5e2a45affab59ff7b6ea70a6950c08791cc46615adaca84250ede6d3", "подбор", "разные"],
    ["f9d68cd800cf820548c5633932f5a3ce46bf4f6147d06a87dcfb1d1e64a02758", "525d674cf5824ee3bf87463e565a9ed7a7cc8e9943b7bee5147b965259652ffa", "подбор", "разные"],
    ["47419c640e3338a435ee572395a793eb40b713db06696bf953574eae95864989", "1d7035337d318a00bce16789584b4f24d35fe02731e2b3bd2193fa909325b1d1", "подбор", "разные"],
    ["4d6365b72c7a53b1abb8f6968c1683b4101b9134ea3cf4169057e53f140d3f43", "007fadf46e1de3fb217993249383d21a567d8481b15f533c362a628960a2e549", "подбор", "уточнить"],
    ["16cdd7b82b9cb7796e437236e34248cd54b375a00af13d0738f411b7c312c73a", "a813e471248d2df02d2c2af6f7be1c9c0201e2fce1a66b0d974b18e7a3cb72a8", "подбор", "разные"],
    ["d08c03f5bb8b1922361a350b30013e4314d424aa09b1984c0e3fdf5f6cde7b0d", "734192c5bdd314b934c370ff579efaeee236e1a77da766228df0a75046da112b", "проверка", "разные"],
    ["1e370a808d925bc6923f79f2cf8c12e8c7338d0140bd8df2a2dccbcacea45c79", "613fccfc57543705591461d0356cd17d9248dc4d24cf68dbcd515f339b455ee8", "подбор", "уточнить"],
    ["d00d8518ea6e5ee894a56bd0cd7aebc9159d6b208e8b8da5874c9d9a59b5e79b", "e983bed971591045f1304052734edb7212afc75fb634af5d06caebf5fa44b1ad", "проверка", "разные"],
    ["59e30783df6ad7b0117b3d4180269ecd917f32ca87061cc064de8f09c4188c67", "587b52733d3444ca1356762b5fe55124d39820b517eede0e6c50b9c313114477", "подбор", "разные"],
    ["00b1f8d4769c0930bcfd6df415a8fcc743e2cac505e2af7c2a886578caded02d", "021b15c304aa87890e85f79d30e210f596cca6c1edbb67dab68471fad7e33ddf", "подбор", "разные"],
    ["2a11033b968eb7e668b236fd8d7b2c1acb14939778114c84d8d6869a61db3b36", "7bfe766ae2b5b964201e3d7899062bc04449f55b0f0f7c384630c70dcc3ec69f", "проверка", "дубль"],
    ["a4fda17c8d48411c4c194d5b12ec39ad04ae8cc289b5ef2031447898146711bb", "407783565553bdf0478b5e0accedf31bac9a54713d7f02dc1308f61c498a7cfb", "подбор", "уточнить"],
    ["586cfea9080fe3a6acdda2a26d4908a9eeae7b534b4704f4f27b4b54acf1c41b", "5a6b8ba0cbe808b47c1bcf02fdf1e8114c7d9ce176f232026355be3bc7452218", "проверка", "разные"],
    ["a6096ba60e6f2a71ba30dd8c93c0499118cf1c8d4b3fee8d3fc433f915178396", "43376a28a8a0211d5ccf44004d074e3f9497a6e69a25183f6c2c5c6938ace533", "подбор", "разные"],
    ["8764aaab99c87539bb57bcee6334dd9d17855afdef953745ffc69685a306ec61", "8ab1de3c683aed97f96e81f9297f72f4bf5f235fe55ef873e81ccab7248176c0", "подбор", "разные"],
    ["dddd0adc1ee59bb91857f92ff27603d8ed73e8fed88dbea2e97f6ac7fed41d02", "43dbc9912a7b6506a06da4bdb865ea0cb732d62a2e25f80d8870d64ea9675f66", "подбор", "разные"],
    ["b7d43a4c9c5a9c6912d0e6ca9e7f59e1737c5681623851c68dad22fb6ff3b664", "9b64181b49d8ca8056e5ec8d6eba2e8bd09cd8f737b15f94a0ef246b101482ea", "подбор", "разные"],
    ["7838dc1ff748060688854308244342b75f41d33d4a6a4019b86f53478b14e0d8", "b19d3c35421ba68c6e15bd0ea186316af957aca990d127708f2e3c4b0cbfa6ba", "подбор", "уточнить"],
    ["b89c5474cb8452ffb6d41d14a2e158cec9dd6e18a20e88d928583397636a42da", "c854958c5fb7986327abd6ec0b218b61e3b6d13abc3e67238b3d7006c54bcf87", "подбор", "разные"],
    ["70c65db4d521bb18befd07c9bd48de9c392d146b2f42410c926ef6932999b221", "3724d9c7b35a0a1da97642365521b7ee1e46233549a75827ae5a5386f024c29a", "подбор", "разные"],
    ["97898e445088f50df3cf4c38c1daa0f2e99c9ecb5a7a868316539c7d746a48b7", "ddb5bc0960d81dd3d464423991fcb1cdf498c3c3a66afa2f25dce55c8508767c", "подбор", "разные"],
    ["a17357f5fcfe12234009a7330c8ed843152419e4bc3ac83918d1cb4d91eff2a7", "19e54d6edba62bee36cbc45fe91090ceba0d021d59d7f22f32396a9291a7063e", "подбор", "разные"],
    ["2df285e58e9efae3073d7ec60121798a4211b998b0ce0f002094ae33af2faced", "758e0cb012cc3e526322199c19b429bebcdbf418929f4e94afe54715c288581a", "проверка", "разные"],
    ["8b911fa2da8b541f86ff927e54b6960b4b07eaf088faaed1c4e0fc60f334371e", "ccf9495e8324d79dbbd10e313644de697152b3d5b5300836077f310def50fd57", "подбор", "разные"],
    ["f1c09a969ddab9761ae6e88bfede6daafc26c8999ee510a237748dca965394d9", "05849cb277d96ac12c2c1e7c2964505066df88b0cf4d5a02500ed8034bc5dcb5", "подбор", "покрытие"],
    ["022eedea415cd444b1a7e9def55710579ff6e92a6e0d090ff6bb43b98c9c06ff", "22682fd53c598876ad41486cbb2ace2c439479f13efd29123ba81af7356f6050", "проверка", "разные"],
    ["6b59aacb61aa977461ebf83612eb2ed138cf142bc513304c96fce5a463f9e94d", "be7a492739468f616d02186a2a4cc4631facc3d2393770090455cd4db40c61d7", "подбор", "уточнить"],
    ["16720d3e540443670e9e1ed01bd52233cfad6bf80b6705652b0e73d181a29d71", "e86d015598b946900702b91e100fa5cf7f1a1ed0af0af5e8ef441aea372118e8", "проверка", "разные"],
    ["1d68b07c1ab130f36a3c5ed093c448fe083d243364e12147755e0bc9fbae0885", "bf45916dab7a4f48f681b83e2d6da75acf9384a3f68110a118527f7fcc6f3e46", "проверка", "разные"],
    ["78d32c9e96ba3e5bcb8e2eee14d55a42e4e4bc6b112e96c935bf77f46c001dbb", "c693d0a10595b3b89314c2c9110162ee6477a2ab9e6dddc281c2a5677073934b", "подбор", "уточнить"],
    ["d34faf9d553470a1e9153bd9b82a6f96b6f1cf266e4132bf7c6327026470dc2a", "35a29e14d1660e1376233daba461a2df573875c7df65539ae47dc53b053dacb1", "подбор", "разные"],
    ["8a54e159ba9340356e09918b8de288ca200c5c57ad388d6507ad357b13e7ddb0", "014942ee9478507ca506ecf037fac959dc3649fc55caa124e23609a2e8ad3fda", "проверка", "конфликт"],
    ["41f7a612774d33a44ae9ff51153984c58caacdae8367317c76e5df86afecbf56", "40bc6cff2a9ec9ef5c0951e109330dd2b84f96b4a42ee829b76b2ac02764395c", "подбор", "разные"],
    ["197110be14af4cd05d80b12a4053a0cde7a5c3dc3d52b34ef06bf5ee2eb4763e", "a119d0d67a8927830cdddf2a2e7a49bc80f276a347b7405764acaffdbbc5be28", "проверка", "разные"],
    ["61c5aa2244844bb49c6e226a93f69d7646304ee18cbdeb2b4d844dd37fedcb53", "abd633d3d762de258ef6258f7dcead58c98b78176fa15ecc888ed94eed0f87f7", "подбор", "разные"],
    ["7042430f80f9d9b0b64a64871577f68530035271f674ba5e873e013e6fe6134e", "0338540d99295779ab4301ca9ff2b792994edbfca5c23b0a7ffad8b37fa88330", "подбор", "разные"],
    ["7e2876df233a8007e4f4a690976410eaadbd0cf7e76e45dcc69da435a3826f1a", "c6a8abcaa20e1893dcd47c081561f9a3591705b7e48628a5a08cf2c59170add5", "подбор", "разные"],
    ["30e3529e2501d23257dd8ee56d4efb9cf0459890668f1e73320e09151d61970c", "d3a57f52244697f50a905a996b7f6691bc23669f8ad2bc00fb9c2cf9f3742c9e", "подбор", "разные"],
    ["8f1af93df601359ce0b47fbbc975668b0f4b107ebd9a5ebd1bcd64d5d3e40144", "3183fd0bbd4829326a1998bd886c3191c7f832f762c69864d60a3923ed01cc93", "подбор", "разные"],
    ["e6d5a9712818fb06485bf563feefe0ab7bbc47ad4edb832258a74677c9ec355e", "7fb73d9da82c4823f17b053afd1fa07d1869bbae5727cc76ca86d08ca3b6169d", "подбор", "разные"],
    ["c7156d44e515ea194f4f694893b31fab45b5436f0a2f1804cfcb29a7afd8194e", "8120f20fecba280e3db4612c63ff6d0d1bcc7b877c8722e3914479599bd96466", "подбор", "разные"],
    ["ec699c95757049014b89f30b3cae993c70ad958027220a6e92e0a33cf0615a59", "94d05b4711f6335124fdb7212b90871e39e0612513a685b844f7d05a53043c49", "подбор", "уточнить"],
    ["f362db61f7e1c80a95825e844d2ca81d93bcab1ff280594d8e9210a184dbc9dd", "6ad71963e67671996103fc3489ce0d42f37c8c30a545cfab8b5b8b978e4f5dfa", "подбор", "покрытие"],
    ["53fcd1aef9a5f98cc33d0fac21e6ca208deb14c3781ad69abe7780a911e18cc5", "89444b00b182ab5fcb96dd8142c9dc154df924e543a708d2ada818642d35fad3", "подбор", "разные"],
    ["2de57679b56ada3ce687e58c33966c40dad439ad9e5f6eb04f0ce75fb110dd39", "e529c61811bf18edf3fa190f90b4fc98459af1e2108423393d2d869ca7b66556", "подбор", "разные"],
    ["12bddb58e19b732c44590324008b9897ddf43bdb687643887986e90297c2aed7", "fae9541bdb743b0c7e8743e52b9bc9a7d0d60b191fec8b7408bf802492a4f685", "подбор", "уточнить"],
    ["d6c271315a25686e0d5a8e2dabb2356f846186056b84bed2e757232250b197c0", "7e9dc2c8b025689a32e2653623e7b3673a5804083298c447a4c56b0446d8b67e", "подбор", "разные"],
    ["35ca2e09b043fa9e11100b427ee17692a3a358e73e7d2ca1f2ea6d78ea6b1a1a", "ce0de23031ce4c6738235151980e9eb65294148e49a643a3c2f5e73c17bb07d2", "подбор", "уточнить"],
    ["34d4b65bff0c3882fbc2db5d300733e6671e11c1745280861eab36d0e04a0642", "be09240cb06a6981a445bfb0a8d96397067bb3a1325fa5798da9a99936a093f2", "подбор", "разные"],
    ["1f4935e1ae5754f99ef329940886045d0f5fd38d2961e54a9f85d821f24f3288", "d49c6f76800dfe917b3af62f47b43385c2e54ba3ed12e71974c63f350a7d3df0", "подбор", "разные"],
    ["3d9ca2d3f3205ab8f55437fc5415a473d729b68a99797337b687c57f9206ed36", "0898b447b2859fdf98708def1891b8387a9723906d64c6ae672e3f97b224f55c", "подбор", "конфликт"],
    ["e7aeb5d602677c9fb2b1cea4cd6e3435edf00b429c512e7aab112cf4f8fdff6e", "c61d9ccd8da83a635dc3795bf5a8e429c94ed9d9f18cceaba9e58cc2c9fff652", "подбор", "покрытие"],
    ["c37b2474b24a985fccdc3a965903b7b3491d9463d68d9eb384e65565a4631f40", "2c9f41db0b8ebec55ffbd0b43bd4c59a892c4c41a27c6f3a78f796c3adb4310f", "проверка", "разные"],
    ["48582db8beb44b51e55e4c078c912ff91f40562ca2e3a9aa822fb83405a72111", "c0bbf59eefef570ba5609addc6cf39261a10d74bc69907f3e8d1305ced2b6934", "подбор", "разные"],
    ["8dc5d5360d7478160fe9d7c58b1fea3ec9f46303b03eb21c9a7c015bd770fba9", "26b15e615d8cb67a929f7dfa7e8aa0ec74f834a35227b58e51bd17716f1473b8", "подбор", "покрытие"],
    ["205b59318bd97f14ad1447e92445cb4b4d545b01bd197cad46f0f2f35e295bd5", "0aba6a67b5d2fd4625714a9f33cf001e37f6a6f7c3d909652c7e565e759e9cb6", "подбор", "разные"],
    ["8334273ddb40c5593edafae9cd437cd1e5532cbfaed611c07fcab8432a2f7eb6", "0e83cb6c15a53cad3f3aa5914ddae77fbc16134e409b59109f30e210781287d1", "подбор", "уточнить"],
    ["2ce36a422e3bfa92c9e619d5b01470dfa0e98faaec64f0fc1d116b90624d6c67", "26e296de7c28e0c3b0f3caae66a2604cd0d97ea74815b1d23d4c774f7bb7d49e", "проверка", "разные"],
    ["53af3830261789d2d0feb79b6bf987bebb1d72f9b2fd64659d34963212a226c3", "3a6b5bca5563a725bf0f409909c425beb31a0751cf124770f2c1fe105b2a543d", "проверка", "разные"],
    ["b7c0e97c69d279ea8931c6bc1ae22f582eb0393a012c401d50daa0b59635171a", "c0259b23313a82e40b2feb8d64d680740b32325b581e0fcd1ee40794f96e0c1b", "проверка", "уточнить"],
    ["601f2935c1137080df78387f7fb963fb6d7b6241e41addc505c1a3431debc86e", "10a1fcab303b2e53d95298708acd2f313141295f7d1e010940321658a53121a1", "подбор", "разные"],
    ["1ad73d6bf2be195585c2f48584c7899fa0da47b1ace646810841ce912b87c6ec", "55890d5b66ac89e14acb66d2b272d80c9ad8a835ae24e9a4571f1c4e558589a1", "подбор", "разные"],
    ["ae0a312bc543bb3d2616af0f57753cfa78652cfc4b0683176e1e08c587d662a9", "83598c1748bdfcd12c3db266d3fb4cd5fcce1d79fcba2f8525bff72454e6e596", "подбор", "конфликт"],
    ["1d8f6fe8b0271ec4b4baeeebaccca066c01655d2a46a59164d490569ec9155a6", "64cdf8fba36aadce813ceff475cd9fea9a33b08b1bfff643ab1cad764dbf5893", "подбор", "разные"],
    ["d8ef56a6ec46e0f3e45ee144f8d9a3989bd7959aa20e216e62e6117c877fd57a", "2c7afec2c1d5f3c43bd3c0f56e0e26d79e61084980d24a3b0a64e7b348781ceb", "подбор", "разные"],
    ["4db17198f6c7c1bf5d64164ed82845b9b16f40aa23f596426a5118ec37a45e23", "fab181e9bf461e06d99a2cd8a9ba2ca713c40f34d1460906ce761999ff52d73a", "подбор", "уточнить"],
    ["3d8da66aa2843bb23060a4529648287855d312e47cf3feec43aa68092546c4b6", "d2f51faad0ff018d90f567b5993b6cbf57f15f49f9fa27eb64774039f9d45356", "подбор", "разные"],
    ["246dd56ab2fc6103783c4554f4ff6848612d70eff40249503cadf6a5e3c4991b", "d85a690833c34daab4fffc7c2c84872f45fcbc23be2fdce7d24c0bbd53f8d045", "подбор", "дубль"],
    ["621daae548fe7db68147218b7091c6ec61bbf8a56474831d8c2c6497dc3128fd", "12ac920526dea9464d71fe68398e2cf0c16555a8baed5a201c3d642b8cc9c40f", "подбор", "разные"],
    ["eb401a2ac5d195d066c35f051d9d8ef5c89af2a26618001d8cd0d4033b12128d", "0604ac32f7e1607a0713006e088f555bfc3c4463e2ab94530e1552524d5c69b0", "проверка", "разные"],
    ["9619e04bc5c4b9bfffa3e5e1420e814d18c9767fcb9297b2527e77d81430cc14", "b70197560fd8d687f524635d2fc29f97c3d55876ac42e3b833ba19607f04b0c9", "подбор", "разные"],
    ["c41a23002601869759aabb1260326c3bb768b022c0e640ca0a92fbe20e900218", "910d5cd2f724b970732ffa4f09aaa48249b6c40f4284375d11388a63b6e84fa2", "проверка", "конфликт"],
    ["2de353bb90a7ba0052749b146f3e9e260628edb5acd30481f8fce57c626b5ce3", "456ffd70db67652856d66021ffc0d5e26bd9028b48daad2d702fbf82a30327f8", "подбор", "разные"],
    ["f4e7cad83472a7e30495879f81ce4e431d32383c0533fb41672fe80c7646b008", "690453810b8ef707e86d1c94ccece6f26969f5e172123bcf8df6a0d1b216e78e", "подбор", "разные"],
    ["822bd7e8c184c48456b2a249cc7997ae68de90faf16e3973a295713659ef9518", "8b9fb779e0b9975b3da8387eb856d20872aa695f0b11ed9956404eeb7d5ac9c0", "проверка", "конфликт"],
    ["79ae93f5d0152a486752c3487eedd032af35983e5577c249c8fdd009276d6c7d", "a8dca45bb421622cb6c1c9b68ce790a0d83c5c00a35e6d01c06230338d903411", "подбор", "разные"],
    ["59ee633f93aa32be8dc31e91c6af624d20bf03f7e75271376af3f9d6ad3511f9", "58c77b3dd4d0935b942b0d54a465ac70d421ec9634dbb084b9f3ef4e63c03313", "подбор", "разные"],
    ["42f29d29ec62c227c07f4fd5e4f58c78b8cba9d89a00ff7f9c81df49aeeb78d2", "66c9c0cb08b4934f67bdddb9d824e34dcbd6dde5689de0519a446723c8d03ac4", "подбор", "разные"],
    ["7dbef56274d3a66e40c5ec84fc0811607c66dd6045738e678bf8be461a63f7a8", "a1360cd2694ddd70bf24c83149910f007979a5dc74d6c320815111fd634b0d2d", "подбор", "разные"],
    ["b101aa7bd86abd9afcde211c0875bba1fb0f16f155ee87ec20b6db4c6a9ebd52", "3c5206f1ce3fd07c9ccc4f949e41ccf11760b81e21f1cfef82efff536d5cc1b6", "проверка", "разные"],
    ["4b26da4f4fa1b35b9251ca88040c7cd99d584d1a2455cd3bc7b66779d3608ab4", "f133f796365ffa15641e521286af0aac556ba3b809972793b6c6e5694a6f8461", "подбор", "разные"],
    ["f90e0b3509ac3801f794427e5716ca5ef488677dcb0ccc94da9a8742da42a424", "8fd4e38e674381c8fba008ee7996a8f12b18c5a2733d19dd5cd9401de6b0040f", "подбор", "уточнить"],
    ["94c323670008b230673208589b3849072553fc0ebb50d2f0374bce0c3f03cf1b", "28a2e4f16a684d48a75a97ede43aaa1fa12b905bfea00135118388d23b30ca7d", "подбор", "уточнить"],
    ["41c4e46f7169f5b875cc0d09092f31dd26fa5804a0876f89b28ddfe859ea08e7", "4128dd671a545a7bd024eb03332af787e11316848bff8200546e03ad9602f7b4", "проверка", "разные"],
    ["33b344346172a8c54737f02f83cf20e2d0a04095549b46155c8ff8fbf861a8da", "370a771d16de849760359a017e4337b0e52d2d051752a30808b7ac2336ceb1cd", "подбор", "разные"],
    ["81ad8e649273e88b8cf24f95a8ad9fd93558bd11e5016edc7412c2a9746ab8b2", "5bf2ae0cdcf79690a456bd8914248ca6edb3d0cca602fa27532595099ac61fce", "подбор", "разные"],
    ["72e880c1f6fadda8711f5c3ea4dc0da28b050e714e1041f68642d8f2d94d2b18", "236ede821b0e21533961d5c7601b1f9aa5539a04e6a20b4b3a5f4bb022d936fc", "подбор", "разные"],
    ["0b62ce2a006b92ba7ebe7dcb1b732c673542d43d2f2a26e11a17d04f825dea89", "c58250d8c986b09483ffdeaca22b17e2df73be42a43f6e055d8b1cd9b264a31f", "подбор", "разные"],
    ["cab1d109bb9aefeb092d2173367a4f227c7bc68da4af3deeaa71612a68d36221", "5ed09ae5a2ea93e66fdefee021c381937e7d9c17e86fcc98e241b13bef706ecc", "проверка", "разные"],
    ["4c2b6f93b560b1ee323c9031f6675d4c438130499e8bba195bc0589bad0cfe85", "ebb32654f37df68c09c4a2eb33b99ad5d5ccf6ba781b133c27a6e02053505995", "проверка", "разные"],
    ["4175d9efba5a926480c9b8b0ac1c57d3455a4dc4a66d03415a29125b0c6078fa", "8dff56e7a4879eb74a712d7235421cc33960bd97712c6fce362ce91c6ca14c5b", "подбор", "дубль"],
    ["6269943d185d8b916702736b53057ffcddad43c28aafcd575ed67af23af94f75", "79a4348edade9acdbc0ca9ef872044d80eed4d658c0649bdd36a62cf2c463041", "подбор", "разные"],
    ["aff535eb3a3d9b3cc4b19a0543f056a17aad1f805fe77dfa1a2b524dc8c3db99", "c8d38cf8d3a39382d3ad8520ac839d5e123933270c4b5caf173c7e77e074f55b", "подбор", "разные"],
    ["b77d9ab285384fa45829cb8be548560e4ce496697f86cd54ae7f32ac4c820b49", "e661d557b1e92613431551bdc274f4c50b37857e32f07367f551437140990a12", "подбор", "разные"],
    ["b77d9ab285384fa45829cb8be548560e4ce496697f86cd54ae7f32ac4c820b49", "f14019ae3fba8fe8f78e12e70d66965920df8dab1d6f6266e4aa8f0839241e87", "подбор", "разные"],
    ["42f29d29ec62c227c07f4fd5e4f58c78b8cba9d89a00ff7f9c81df49aeeb78d2", "ded86918611ae1a6fb325eff9f9bbdb389bc4eae5f7d9462762343de77125755", "подбор", "разные"],
    ["42f29d29ec62c227c07f4fd5e4f58c78b8cba9d89a00ff7f9c81df49aeeb78d2", "fa02c728be6d274baeb87f92d20767aadf257fdea727ef3ee9e1d7da8bd662b9", "подбор", "разные"],
    ["42f29d29ec62c227c07f4fd5e4f58c78b8cba9d89a00ff7f9c81df49aeeb78d2", "8536793745dd623fa9b8e1a62ecc92811bb6999b697605c5da4accd594abce5f", "подбор", "разные"],
    ["c031525473c899a1b9e340eff26e7e9174813099ae3567946f546d8312621666", "a72f0e2db60868ad1eb1b740de40afa1975cbf934a9745d3078e7fe15b9b2a40", "проверка", "разные"],
    ["c031525473c899a1b9e340eff26e7e9174813099ae3567946f546d8312621666", "0d235de1c99658790b89e546061e2b7a0db7b4a68f321a66b072cb035084aa42", "проверка", "разные"],
    ["c031525473c899a1b9e340eff26e7e9174813099ae3567946f546d8312621666", "26fe0f9e50c0dd41ed02b73d13cb2b0768d956a51fdfd38447f36bcd38ddfaeb", "проверка", "разные"],
    ["7dbd90258d8a4bb99f7ff5ad4d3805c9d06bba1d7a81037901f17a957f61b3ac", "e836ce1a881d42807c0bd58c352a2397a9dcdb0e29407ef2d2b7c7912108684c", "подбор", "разные"],
    ["7dbd90258d8a4bb99f7ff5ad4d3805c9d06bba1d7a81037901f17a957f61b3ac", "c0f18ef73bda27f6b19e256da6dc012c793d2b2003efdb12186562398830e9ac", "подбор", "разные"],
    ["7dbd90258d8a4bb99f7ff5ad4d3805c9d06bba1d7a81037901f17a957f61b3ac", "374686f0af63e2b0c0c7ca15c58600307d9c525554a14c790e48e6b32ced82af", "подбор", "разные"],
    ["03f22a55bf80947b8c09edb497d88feb9b8f16a7b9f3b26c3c8a010c1338b121", "64be62938ebbe6fa0c73ece5c9053c2498853918ff657a97c7eb0728ed5d9d92", "проверка", "разные"],
    ["03f22a55bf80947b8c09edb497d88feb9b8f16a7b9f3b26c3c8a010c1338b121", "70096e8c8ff7066e062d10b7d0d0a2b2c765f91157042e6a7fd37f2639002f09", "проверка", "разные"],
    ["3afa282819b79e571f9d488b5b57f5b8b1846db7534e25a9a5ac363e2ed94214", "cf1898555c055ec2d4d5855228e4b6367888e81b1cdb07bbd5f57e9182d05429", "подбор", "разные"],
    ["3afa282819b79e571f9d488b5b57f5b8b1846db7534e25a9a5ac363e2ed94214", "70c65db4d521bb18befd07c9bd48de9c392d146b2f42410c926ef6932999b221", "подбор", "разные"],
    ["3b9075cde11869863fb408e6842669a443f1ec870ae21be63bec380093de9f4a", "bf2f4130814138f3e2c06e1b56727ddf7a183cbcd78773b9a3a56ce0143d84b2", "подбор", "разные"],
    ["3b9075cde11869863fb408e6842669a443f1ec870ae21be63bec380093de9f4a", "de4bbde78e0dc63b6d7bd7ed50b8595b5fd5634027c45087ebf1c33822479575", "подбор", "разные"],
    ["266851c0291f62f23f89fb4e027c7f0b3611110ac4474473da2a4b09a8a690a3", "2bdd38426fd1f55dc66fc92307cbcfc81e68ae1978cdacd64c13acda9e2fca2e", "подбор", "уточнить"],
    ["266851c0291f62f23f89fb4e027c7f0b3611110ac4474473da2a4b09a8a690a3", "6bfb79d3e0d4bda5dc193beadcfaa077a49d55b31417190f6055042134cd0e7c", "подбор", "уточнить"],
    ["7c5dd42d5eaf3220d48a3ebfc796c591f8f4254bb8fdd9321bfd32de403a01a5", "ca3bc9bc2dfc0ed68dcb737d2ef3798cf0f2ac28716f312c1283c7f7cffbc191", "проверка", "разные"],
    ["317aa2751c27bad07394a2f9d279eb443c9b4658c273cb3cd5b87e25f153dc7b", "51342d356ae4d67b76ba2ea95a938cc32431f1aa88cb459fb6a0ca7e99dd8ead", "подбор", "разные"],
    ["fd63aebdb79770b963b5ee81cfab4720acf1d00457289c63f3622c4a26bdec6f", "41541c94ffe0f5a78aa16f7c2cf3b8790585d936a4d7c72ce88673b7b1c44637", "подбор", "разные"],
    ["fd63aebdb79770b963b5ee81cfab4720acf1d00457289c63f3622c4a26bdec6f", "e0a9928549b2a3aba003ef9f3fadf0f586639e1c3281f1929b9672edcc271c72", "подбор", "разные"],
    ["d46076806545bfa65adbe69e7ef0a40a58bf30e1bae9207552c055de94502809", "76b7d250a17ae911ed6350c31f4df9bd1b9c867d93083b0dd5537d0c5a97b74f", "подбор", "разные"],
    ["043391541005779f2b8a851ed14e838eab442c364e801fb86d5eb8f3a2275cdc", "a86609b6cc9c43f6ef8db6be672eb29cffc44d094840b11108f31900c9fdb4cf", "подбор", "уточнить"],
    ["da6a0da15326f6da70308ff3f98b9d89d1443fee6ee17f814aac0632809f8bfb", "b31fb36f17a489561700b16ac76693b9b556cdccae5c023138ec15ccdbe54aab", "подбор", "покрытие"],
    ["da6a0da15326f6da70308ff3f98b9d89d1443fee6ee17f814aac0632809f8bfb", "64578b3788fe977ba2400962a2d948a37c04429304bafb8969c6cc245f2d8a9e", "подбор", "разные"],
    ["7c07ff48c6084d5e3b8647fb35192cd0693f05280371b7795f0bba2ac6c291a0", "0268557cd7c56df3e7b2276abf2eb702a21e4fb5a262c05f38a12be23f860048", "подбор", "разные"],
    ["58338a07e35c2770e5f874712492ed7fed22d77564f848b6bbde3c329c0de9f6", "e3cb8c289a1b71dad87ffed0e0ee6f06be520442ab125087a75160a646534cf2", "проверка", "разные"],
    ["6a77bd42b24b9b4dd9f3dc39efad312a474d08a9b516b6113d04a4893a6d9d42", "d0a8ad1c0dd66a1b3e16f03bd2480efd78f1074e644d10782dab849e1ea78e8a", "подбор", "разные"],
    ["618022e312a4e3100ef569450db531eee55d1d7b75b6343c7abe2fb56837cc5f", "8918e4319e0b938f4d512fc98bb715f2667473d597a2b7625ebb4bc903751e86", "подбор", "уточнить"],
    ["1feb3b37f950ac9a3e59d819c3aee9f643dc033619a0ce9ff220ce87abc5936e", "8b95261e22a70af25c7b32337de0e6ef1dc5e348693d02bbe43b1937c20c0327", "проверка", "покрытие"],
    ["a8dca45bb421622cb6c1c9b68ce790a0d83c5c00a35e6d01c06230338d903411", "3d2ff6920a6b10317ae61df6379cc363fd70b8adf7f50367c4f5700d61350068", "подбор", "разные"],
    ["7f99df668d0a250efc1b85bb128bef1f4dbc7b5b08f46b359f4213b47f4cdbcb", "a119d0d67a8927830cdddf2a2e7a49bc80f276a347b7405764acaffdbbc5be28", "проверка", "разные"],
    ["648002afbf7f7f4c66652d4947349e91459fa4bdabd093f12ed6630d2e81d2c0", "81c3d10f819130185750f2a54e01801ddf1fc7f9df348e6633931cf6c9262999", "подбор", "разные"],
    ["fa3f0ea17d0a6f86605fb840d2a09193c70289778d7ec93974e408e873b6876e", "51adc792a3e21339585e4acfdad0dcc33e7489dab461829d379f371b6f37b5f7", "подбор", "разные"],
    ["a8fd909e906030423c18ece755e588affbd9ad8db04dc854cc6686339718b9b0", "1bb6e37a084de57055f77c9101926fde98ac7cecf6410f6b48bb0057a2f6ca13", "подбор", "разные"],
    ["c4667dc1c4d84972357806d35bbd096ab25a04d8f851da554ffe30aed18114f3", "f2ccd49438752f3ab7b634eca5ca80299aa3ef94fa61d3b46137f53924d7fc56", "подбор", "разные"],
    ["0b62f578b800ac0a20507d56741e6b1387a27f2cbdf1749a96b1d67c4621e437", "fec25c18e631a65f9920f3ca3ea5311bb39122864780074e93c4592d6ba4df30", "подбор", "уточнить"],
    ["1d329899ad3eb6ea75a4b07561911931c35e2a871f99964e44e491f6d3153559", "14d4544c93dfff8cd0523d4efe9271c3d835049db64ce169e8d2c5bf87c293c7", "подбор", "разные"],
    ["5edc83ed478c6570d43a08adb6b952b4a97fbde1a36a55135eeab7ad0fc196f1", "335bb5d716f2181cf985d9339d9f95bfbf21bd4b84c8dc6bb685f792b554a135", "подбор", "разные"],
    ["05d830398a0d1e810d352fced1a45f29ecc47c675f6bc5594ef7d1fd20a32ad9", "d5f5b33019410c758eb496030b70e4d88bddbe8b2cb5b8a074d6db9ab83e7cbf", "подбор", "разные"],
    ["337d74a2e98db94f5f59d4a01d8542eb7b0e4fd7e304835f8e931b39401e1b56", "dc9dcda2fed27b8bc25a44e970f19ae45691597451923513124a932f509baeef", "подбор", "разные"],
    ["44d13fd3a376d0573958da393bdb43231410523e1c669b4f7a27b5d46c582c56", "09224b38924b510c611ab44e7828dad77e4b9fa2dad8a060c948d001928b8152", "подбор", "уточнить"],
    ["b045b43f6b5b586ef655b699a02863b767712de2eb904ca62f3248ba0f841e92", "355a5e9c3b64e65a296c7e7149a71aa22ac8dbbd466bbb8076b8e316afd2cb3c", "подбор", "разные"],
    ["908dff3715db9e0ad5b98677090479981817fea43f945fb8de2204da1bf83127", "c807840abfc765072be28ca77537812ed7dd260d5a90aec093a557d01e98e2ec", "подбор", "дубль"],
    ["d8b42a93d0b75e3a2391a2cb5c9120be3a2a5a18b1da1a2bc171b250b8560cdd", "000bc58c776ce37666d42be6a29cb436a7ba46d026484806664790d639c296a5", "подбор", "дубль"],
    ["0dc437ca6bf7b4c7bb3e8edcdb896c5a27f17e6c585fbe27efd15bb650dac53c", "1e4f0e749570ba3a3e80974a638787d9519ee37db9e75415e927876f8d57f299", "подбор", "конфликт"],
    ["ad7042df0c183b0e0c8ced69897cc52d0eaecd54b2f8e376e11bfa01ba3df689", "cc24cc201d9f3886a1f7bb68dc4f18386046714b90244819a74daf3df5c2ea00", "подбор", "уточнить"],
    ["eacd7a098afe2b00309123f520e71af3fe0bce87cba5dfe5708118c41c184579", "6e4aa334604e16884e44eed74a3fa734b059f8c0a559fe5fc9a87f404021d040", "подбор", "уточнить"],
    ["43c49bfb96d556460df6883e809a25e230bfd0910ca9a397dffe2f8836fdf959", "39f20d6faca4502373a0037b74513e6a39a0b981c3f36cd1751a90059eac15d9", "подбор", "разные"],
    ["618e1dad2f627cd90d962c65be6703ef724446cd49a6635a2534311497bb2efc", "06805aacd0d9f1275aa9db4ae0fb35a95fa7d9cc223da86075319213113085e1", "подбор", "разные"],
    ["edf84cf05f9b0b7c28783663b9c329b5e534d55958ce82fbb85e0b41a10d05dc", "f16041a3689ce55d60d0042b7cf02e3b0be56fe5c52decc196e7e12f9387a871", "подбор", "разные"],
    ["898d0d71fba12fecb199139ab8d466c8de858ba6265032fd11a7f1f7aa3ae4ca", "afe4168c7934d81a401523a78cff1bae1d18460fe600e83763a0e1eb34c4c5a5", "подбор", "уточнить"],
    ["ff60ac2a7b9afd841d987ffefd65e0e36851d5cc9041c4f056fcec1812d1f960", "446260c2713b9d0e1438551a1f4ebc774f88c601eebe9a4103b9a9489bf7fb71", "подбор", "уточнить"],
    ["d3106cf3dfd1d4adf7018517342fa2ed79dad03f46d89f82d325b87254288ce1", "99d2c674b0c3aa84518155bbf7350959f3b3ac2285f983c33ff92f7296bb572d", "проверка", "разные"],
    ["20f0ffeed4251f7807cc14ad1b7992a4a819cfb42f26bdd3ee1a0c1ff8def44f", "b73bb64f4fb2bca3006ed182719f180856104d163d19ee1a93b1f9a25c6fc88c", "подбор", "разные"],
    ["48911173e8a9f32cdc4dc696483bc4fb063a7850852898d9dc3325ccd5b7f19d", "136566f0140eaa0db8f1dd547608322be75e707a7f9cc8ca1892032c602132cd", "подбор", "разные"],
    ["9c87671c9965c191e1d165f90465ca24d89d9f8ae51bd37402ff2bae1c5bfa2a", "a0231a52c0b21d989f165ba01a06eaee072946402e4bcdcf7a2c0263c3fbdbbb", "подбор", "разные"],
    ["17e2831765067e14772d4c28b995f655ebe7d14b26ea5be43682c55fc2754ab8", "1651a1b59f26c72b933203a1cdc31b7a60f54b643fb237213e5e4dea6fe25fe6", "подбор", "разные"],
    ["1b9c257d68969ff540db298c71c8a56f7025945a824782f6752f18fd74b0f44d", "2a386b9f3f9d5fd222159673a1d07a71d8fbcb2db0a9a9bb6990619cbd53438f", "подбор", "разные"],
    ["c04c99b452963d9261ab6d26d93cd8909d57dc93ff4621ce1b60ea4a9201e37d", "c71c207d62f6aae73d10a98a95a3ebb1ceed59279ee2f6e588e481a5c186b188", "подбор", "разные"],
    ["bf1c7aac6da566858de885ba2e09cf654f82cd8f0aa194969686929eb8dee309", "08e94c0638b55123b5b1b0d022ff74d4e97a3ed13c6ca2da43f97a2282a62ae1", "подбор", "разные"],
    ["b27b90a93fcb08129fee3ab28d25d7d3e3386ee27d4f270648e49c7dcead1ecb", "ce033bc9ad3390faec98e80d59730acda70f391e7f43eb626c97671740c3c39d", "подбор", "разные"],
    ["869b6dbffcacdcd9646b2ab2fe875186ec5c08272d80c91a6c90a8a46337d4e3", "259e08d8bd3f349d26d03e188a716433a8f8acff9f403bafe368fdd793e952f4", "подбор", "разные"],
    ["7be19f70afaa7f24e12aad542ff4effc4f84459f214abea1a3aee9eeedbf449b", "6d228f119a414ed132578763dfde3974c6639c279caae994ff6a7c6c9854732c", "подбор", "дубль"],
    ["fc0ff78b8e38cef661e0adf849fe3fd94661be3c2095c1ad6d1ed5d09f6a042a", "d2e3586fbfd914de1328c51040ece6b49fc0ff3ec8e239c1b13f26ff859cd28d", "подбор", "дубль"],
    ["baa9277d996a7413ae540dd3b87e9745e475f313b00d03dc87b55289194d20a0", "04715d057b8528d6286921592c70e84d05d1a54fc18100d0e4a64369e90b35bd", "подбор", "дубль"],
    ["8ad19077d6bbc2752daf4c933cc685b17d4df4f675c169c50d9b3153ecb75151", "cd374d607ecea9aff325af28f97d9f20aaff177d7944277fe2bbffb8ec907ac7", "подбор", "дубль"],
    ["9981dc912aa065a81838e25b15e0c811f4ca35d58ce9273a80adee088f8c172a", "432e0e1cf5ac872e2224af21dacdd87002a558ff364845c88693f9b1c6ef3c9f", "подбор", "дубль"],
    ["356f6318078f8d54a677755854fa530e2a4f98ef27109cda696a72c07daac7fa", "d937045b220fca830f7314de272a2c4380abcca765f71a2cc9b0f6047091651b", "проверка", "дубль"],
    ["0affb5a4f523ae3dd9ee33a3bc3d1e230de426894d2c8a9f157bc20e1eb5b863", "951edafefa9014cf0cbb716a4c7c2e58b55ea69cee77e28eb1c981d98fd45ba9", "подбор", "дубль"],
    ["9d8cd46d5a1b533f95fdc89f874d21247e8e3f8d9c94855c92046a1c1b7d0004", "4df13e6e469a3e41d67f7da833a6560637acec78a3ec9a1ba44e6ada1f4e7ec2", "подбор", "дубль"],
    ["c0c758d429958c6059157b16e7f99d9a5ba8efabd3639ddd07b18675a44ce430", "a4a436a0c4d5860601c3b3bd0e45f82606a5cb9a1d2d5dc84d77ebc7964e1005", "подбор", "дубль"],
    ["9c694025bd48b722bde4f1de2170d51e1b24564cef5547592195c6a8afa2b8e9", "c64cd87555bcc00c58c714bb4f9d1c7c63c88ee7e1b05641ad878869fee2b4ca", "проверка", "дубль"],
    ["50ee871d71ecda5ea277f6ced87c5540aed30cf80bd7b923eccfb68df911509e", "f20d5bf9bfbef68b8623a2a5a4c76fbc6ea59884b5dad9cdc6cfa0aa2d92ed52", "подбор", "дубль"],
    ["d9f719ba8ef0858f5e0e8cd3d6ac5c12a2f6ebe413fdb2165e43994f04e7a63a", "e4e37ebe26ec53df922e94dc421b171f54e5761f01b3613ef8038660f81fd94a", "подбор", "дубль"],
    ["8fa6bd52b6fe9fe05460eff65cc9350f7a74227e20cc7fa3e7bc29c05f423694", "4528974b9ee2c110be57d060c794e4a29bcc37b78b5264517ed453c7f2a4483d", "проверка", "дубль"],
    ["e0c4ae161ee35bedc5699e260e268477d78eac245d8513e7511a1d83606d083f", "449f427393238ed9f41737574e69666473ce5f883a43c41a25ec0e8a98487e10", "подбор", "дубль"],
    ["c8395ecada74eeeb7739b0a27bc8bf016795d5fc66a4c813bab7b2821d4af5d2", "a2142a03f08f0457f187c3e28616156a81206c0682b4af443449a5f39704b4d4", "подбор", "дубль"],
    ["08b8f6cca6edb3f71267a06cebc4c0cf9888587deea88789820e437db5f0b22a", "ea39633192050568a0cd3b3fb2588170c6e95a90290a24db43bf8a4a1d4c82a9", "подбор", "дубль"],
    ["07f3849e5182fc0d181577b14821bab6bb647d128aaaaaeb547015311e11ccf9", "6b2886ce847b06f9c557057eb7450c759d0f9e0df6b43ea5d58dfc27db6e3743", "подбор", "дубль"],
    ["5e6c80ecb76450290f06d6a7157078ca0902450f6aa71df05d5fcd63e2429cee", "9cbd92e8e35ccf9e6f75ad535e42c40237fbb9916c46b0ad73882b5cf7b888eb", "подбор", "дубль"],
    ["ec092368dd459de5b3682ac76f72817085b26f2428c924b17efe161e526409c5", "a16f378bd174bff7dfa50523b63e3fd13433ecbc77f9b12b2e3d9f17fb55797b", "подбор", "дубль"],
    ["fc12cfc1183abc075a105c69bd65642482ccded5f7c810de97168902f67cf079", "a714dd4b54fd800ec33d12b6c42bc759b24c1ef68e9925de8859b6e7f9b290a3", "проверка", "конфликт"],
    ["164babbef8a1a35f73737d3a2f45a6e75082c8c7ad48d85357981dafc6bdf7f1", "f2b06f7a0d5823c81f60e75f5055b039e0e2d0b7b4d6e67eae9bea7aee597356", "подбор", "дубль"],
    ["0fa0511727b5f876cdbc7ef66c154b6d37b8b3d46f9c7a5ec0367dc8b3820d04", "72429314f580cf9ac1013bbb49bea2951dc236853d51ea1ee3f58bc1f993194c", "подбор", "конфликт"],
    ["e0830c3876b4db835edfb94df1de91647d132bfed2de44a8d9279d20279fbf63", "e9d9e7102a76a7b3582de83be6f0bd6edfba81498638d6a5585818f21247db37", "подбор", "дубль"],
    ["83fd66c68a7421d69ec5c5a0d16dc610324b0f8cc61e63704f9f7655d1305054", "b287486f64f8981ed768eea4ef491504cf74ce9b9dfe5e6b03bc6c37900154c4", "подбор", "уточнить"],
    ["a39082be6180203e47c0709f9a09bdc36f89bc33d2cdb28320633ba5a05f2439", "21d0dd6c63a3a6a872091ae51a5bc17ba04b9ad39139fd33289301c36d1840d2", "подбор", "дубль"],
    ["08d9295a326c1bbbc79fa29e803d2486d4d2e4f17996299a4e473c78f556c736", "ae7ab609cdc1dee6c72510330fc36de0203dc1a7abd8e88adb7cb45e7a2b273b", "подбор", "дубль"],
    ["54c7246378c3c23bedf282df943221b6e237bec160575f28b4a6572ec71a5f57", "fc1319cd026ef419ee0f4be005ff1bb5180d866420902d7c40386b11efdde700", "подбор", "конфликт"],
    ["02ac1708acbc452a9f1d916226aebc0fee408ff87ac8477dd1649941ec740a1c", "dd72a6386764812f77975eb2c4fc61fd9d51e6fed80e11c02182972a570f0a3f", "подбор", "покрытие"],
    ["0a7c09443db23c60fade625c09d5753c70d3db94823686af9783c09387955a7e", "5bfbdb7867fe645c258ea23935c9f844207f86201d646b45d41c4517e44d1e88", "проверка", "дубль"],
    ["d443fe4444b43529abb8716efaa4e645dea503d5c7afb62eb4b91b81b5a711aa", "f4bb43bcf296e34e21c3c7b58161f20461c39e04ca379c3be5ccec85c548d9f6", "проверка", "покрытие"],
    ["e6726d07b041677ed7c08e5577a6b551555209e135b4d5e405912026fc5d5703", "793bb54cc4796647230637995f0ce3c71c74237a4bd011a9898e8d7dae229e0b", "подбор", "уточнить"],
    ["f20d5bf9bfbef68b8623a2a5a4c76fbc6ea59884b5dad9cdc6cfa0aa2d92ed52", "4c00f9d0d115996b235acad397e4870d252b1d68d9d9acd67b0acda2114e46e4", "подбор", "разные"],
    ["06e363feb95e373d1fc9cd6e47877f24d76943e9a4c35a542a101deaaf78a6b8", "a20ee6ebd98ea8766a7b0c14bf19aee5310b9a5f66ddef0c3dae45fc2e9f1c0f", "подбор", "дубль"],
    ["07f3849e5182fc0d181577b14821bab6bb647d128aaaaaeb547015311e11ccf9", "9cbd92e8e35ccf9e6f75ad535e42c40237fbb9916c46b0ad73882b5cf7b888eb", "подбор", "разные"],
    ["5e6c80ecb76450290f06d6a7157078ca0902450f6aa71df05d5fcd63e2429cee", "6b2886ce847b06f9c557057eb7450c759d0f9e0df6b43ea5d58dfc27db6e3743", "подбор", "разные"],
    ["d59362458d40a06b5ea3ba7ba8b3c0352667a868189ed4fcf734589de5380d9d", "a2b21037a7cbbc6e223a67f9f838ef215f9109e89e80438f3bf21871ead0d7d7", "проверка", "уточнить"],
    ["7a329fdd009b8f7a66e5a26db0d5262af7e698edf6f1a3757028078db0762976", "96cdfc00eb3998289b4fb88fd6511644e08d32b48915e4c955bdd89620062ad7", "подбор", "разные"],
    ["41f9e9b11d519bebd7e03ff1c17e28c3dd7e32a1f25b34e48ac27aced07b6270", "f49fa2d906896d55359078a2c908676249556f4dd126d9e96dad1afc09084abd", "проверка", "дубль"],
    ["f2587e3d5062e72f76c2d18bfeafe42e0dbb7630bb950cd0974afd1265c81f3b", "c23d80ce40980be744cf135602d33232766db9bf5cc8b131e8fd7c3fe4e3e9a2", "проверка", "разные"],
    ["7c147812a09cd326a8555cf8e5d69ddc7a7e72949af2d1a068eb0e357a55b0c7", "4afbd3cef756e53fa0b4abfef37c27d247f75f2091e62590f9405cc15408719d", "подбор", "разные"],
    ["8cb2fde1dc18e6919df0f4c609cf0aaa7b11ab0152c020069d800fa1e128b718", "184d3ff0fa1be953d7c85bd2caa75040ea9f90c5e0d59fa23358a38de1ebfbde", "подбор", "дубль"],
    ["e7cff9456691041af9f20a2e67af99e0ed5b20fbabe2ff8702bff18707b18251", "955667ad8166af11b16afbd87009fdba046ba5783d0a166711b35531ea46d0a9", "подбор", "дубль"],
    ["fab1235f71f033e1f54733b1f05e2037e0b2a5fc05ae3c7779fbf29d81de05f3", "c294ab8b0f0c18fca32ce8f59de7c71d6865f8670781491f39881db2b3940b93", "подбор", "дубль"],
    ["f9364843a572515861c6cf4d9dcdeae5892d030ad947498b6ceb647b39c16537", "731a4261caec58cd8038d28308a6b84df104e3ad9591ce08d809acd48ad9127d", "проверка", "дубль"],
    ["074b362420d43b657fdf6b3b589145501193bb1428e2e1ecfece3db088891c91", "2dfeda92ca4f5833d58ecd0d49425103fa6eef41098520cb4c8b41cb1accbd4e", "подбор", "дубль"],
    ["27547cc4db1a29d64e5dee0e3fc7ac7722a1ce90a3c2ea5334df23cd5cb2e393", "687278fead7eaeae3a460adcc6edb82f500bca1a5bf3fc1e7c145448e596f507", "подбор", "уточнить"],
    ["c101ee5cbb509327f1c5666568875fddffd45c00c63efc5dca5a290c8fc17953", "a1b1a556ae146c9147258239a6e45bc4c5c00ea42c3ad48ecbbb1667b665d088", "подбор", "дубль"],
    ["72761484650ba4f4b2346c9b9d02287eb2a3e642e0aaf2485f148d2fb5acfbb7", "79d254ffca5cce7f2b52be2e45f5933c87c34ba108ef84e1d56aad3a178cb843", "подбор", "дубль"],
    ["40d9c6c615bc3819d40c1c0a2d96ae1dabbcc1188bcec0200c6e00b2b365816e", "417508ed7a16865b86637cb71429cb090bab7df2765c8832966865be8d7f9457", "подбор", "дубль"],
    ["304ffb471e305cce916ba2def18122e95af335287a52e5273ebcd2a60cd934d0", "6ff2608bf2514f21b45b154b672fa243cc9d160f9b4014c608a2cfd681797fdc", "подбор", "разные"],
    ["3e25f4b151af9de6c33cb9642094a938ebd6e358ffc4999fd5567ed7a55cdcfd", "4e3e2bdc33cc6d3a04fce25b89254960e4f8d142131b87875a5d3ba40fc4139b", "подбор", "дубль"],
    ["b6f5354d42bbec23b51e3be13dff2b4fb4def36c3846d65f62adc623e9ae88f4", "5a9fb54663bd4327c5aefbf371da1726d972ba8e7cb4aff17341548afc0d0f85", "проверка", "разные"],
    ["61c20f84c9533aff899a63bf9b7c93e3ceed9dd924a1640d4e19f9ae13a5fb67", "98b115f8b5a5829deeb6cae1aff0d843fbf6e79895dab75feaa3a9c3007302f7", "подбор", "разные"],
    ["758e0cb012cc3e526322199c19b429bebcdbf418929f4e94afe54715c288581a", "0dfdd8e7e1c4eb0d1b4287263c527ed0d96d65db84320cda78cd41a66486d05c", "проверка", "разные"],
    ["47bf10089d5e5f3acfbd8478d79e1676644d400f59f45580439055b77228938d", "92c34fa50d4b6d9491768b9e997e5e38e97331fb3714ff150db5ed98eb32c1a1", "контрольный аудит", "дубль"],
    ["6ba4f638f99d2239992becc7016ec28ce3da9f37947372adebdf7a6738403336", "10f03db309ef1b6147bafa512fa4e82717d6b65a48884de6182ebaff1c6e52db", "контрольный аудит", "разные"],
    ["ad048bceb9cc0ac33a4a4b10852ed5fe10c54f087007f14a4d274ae3ee5878b1", "e54f467c005d02748a0db50b71669f06c827dee37b7e9a0bf61ade5fa3f597d7", "контрольный аудит", "разные"],
    ["0a0dad0c5306ab0f1c186ba521200e600014956e915b21470c07198498c7279c", "9b3fc691d6aaa2f1ff09e132a138ee7d335aebed8a39aa85f5bb6b4d41022c73", "контрольный аудит", "разные"],
    ["fc0714a2be3ce21e25f4becfcd3a627767e5e9e3aba600f73642cfe80d6dfd44", "826d9f200c037b5e3738c6fb173c7856e9b62d1257517d40e5a1ccae82a760f7", "контрольный аудит", "разные"],
    ["3c405384a6b29c0e3b02982669c6859027abe244db243bcf055077a65e1aa7f1", "ef618e3de3c5fdb79ef19a0b92630fc62123333bd0361792874883dafc01151b", "контрольный аудит", "разные"],
    ["e97f607950e0558a8428c5a1c7973c10cb1bba690e530478df1128e5091991b4", "0c9290f2346213a3e436e8482e83f5e7c7e2f68492935113566a5e8324409607", "контрольный аудит", "разные"],
    ["9dcfb19da3e976c2c4b88a8e2b6d442110f3586a35fa54b3b32c42bfe94cf5bc", "9cb6874ca62eb3c7e585f79dced7983c029e1aebd79df1a293263f89939bafd6", "контрольный аудит", "разные"],
    ["c38a54564e1aeb52355a2cafe61c3d0b3ccf4120e2d036868541f3ae7afca3fe", "e958b36ccb74431fddd701f28acb9712d15ab8c6217119a14b2216e5c1db7b1f", "контрольный аудит", "покрытие"],
    ["0025e344c20f26e23bd0b4a9c4178f1ae1b4143e848af1efdaa742ec66e22fd6", "f5601f6eb8300228664117ddd09ee6af2df483ebeb43363af631324f3c550c41", "контрольный аудит", "покрытие"],
    ["fbdbfb7ddb65aef66d73617a11e797d1a2a7e69bc6540aa8e6b7c9cea26f034b", "23828ad141edc2ade17889f5653ca9e0ea43de44d6114885c206862ed65f2d3b", "контрольный аудит", "покрытие"],
    ["3f47f7cbf7f311b2d31d9176c3f6765e2eb248af8f43570d3daff4b3b38239bf", "52e51172da8e4e8185fb3f7a02d610cf8b72e1d0ebfc5bf1355212c8a3db01f8", "контрольный аудит", "разные"],
    ["13a35ab91dd8182572390f6d8fee1f999e1fbc3824164b20b99a5959f312eb27", "3b97d7d81de2df48eba009f0a049a26707e6ea9b142e0cc6c2c01bd9a9e27ff6", "контрольный аудит", "разные"],
    ["3cbefdbd966fec11d5abe4b1b4b30b156d3f7ba4b170a533a31dcda6e8b64ab5", "cab84daa4312d75d1e16770f143a2542bbad64f668ccbaa80abd45c4db131ef9", "контрольный аудит", "разные"],
    ["678a01ddb9d7ebaaaf312360b72bcc985ebc5a6b51f71bce03876ae823ad7bdb", "5262c23034802f633da3f2899e40322dae72aacc4d09e3be095cca96c1977042", "контрольный аудит", "дубль"],
    ["e3ecf1d6a44dfd662f872d9ebbc68486786df1d107fdce01a5ad39cf8fcdb6c6", "c5b108fbaa828b90f3a10438e819f4067217c03713bc735756a7943dabc7c70e", "контрольный аудит", "дубль"],
    ["7b7ce04a464073ba0c1ba16dcb62df0a3dc5535c28ae213d2ae98539e53cef25", "8dc7640f3b00f9fd410fc65f5ccaff02153312af58d21eb36d117826b6dde6c8", "контрольный аудит", "разные"],
    ["cfecdc5a1622b53a3b649056d41ae35fe0b5d50ea935aecbacc818fb5d29f8fb", "c752bda7667fa37a507f1fe6e3c9d115126ea056612beffdbbbea0fb418580ee", "контрольный аудит", "дубль"],
    ["2c1c7573661b50c3a4769552f48e70ecb42509537e32f79d85029d49d5511c25", "a8f0d8120b414428379b0eb0379365ba44ca8ad9595bdadc11308d0b7645038b", "контрольный аудит", "покрытие"],
    ["503e186ddb038bf416240f1e90d1c7d74af9f68cebde09691776582ccb17d3df", "b5b5f625ef2ebbba2748c656be6bbf51a9f7d079024399bdc3836dee80d826b7", "контрольный аудит", "покрытие"],
    ["3bef1aec92ed4236aaf0627450dfd3a1a928056f0feb03fd40bc52535a656cac", "17a2f64cafb61f5fa4f0e08b26df731ee40160043622505fb460a17270c34bef", "контрольный аудит", "дубль"],
    ["7c7c6136fb80a392351e6fe6fb5f720d76746b4e48616ff37e11a187777b24e5", "8db9f64872e9a72b7f2d08d844ea0628070934571db66c6b08b7124fd733affb", "контрольный аудит", "уточнить"],
    ["532a8b792368662a56c66842ef57ea98322f94386abc556f6008dbccbcfba9b8", "90a3b1cbc408d459a6b7a40968853b33e8bd84ff512d66dcc9035814fbd8eff5", "контрольный аудит", "покрытие"],
    ["551417db8b1fe0d00691b0f28c88d6706e65f1b07ae9f59cfc9473040000690b", "23b657de106b4f5f02dfe858b7ed57b777aafdc9d6d2d35a20f7eadfdf40dfaf", "контрольный аудит", "разные"],
    ["2fcaf4f99838594d460a09bcc4fbdc6ae135eafbb67050e33aa8b66401e55fd1", "959c870e7decb38fcfe05eb6c36160b3d791b5211ac5bc18ed17300e901a0575", "контрольный аудит", "уточнить"],
    ["6cc4f1feae495425f9e1f854c555a4e6748cf009496872710faa93ba5bf78da6", "2d33398be35d16d71acb87ad97d47a449e1151bcdbac063b7ae02cff59141fe4", "контрольный аудит", "разные"],
    ["e56bb96533705fa06db650445cfc8e202bf0b97d53e3f021152a7ea8d2b7b6dd", "71a11c7aef6ac3ddc6608ca922b58aeba2c40c49406db5ebdf1aaf4c6781583b", "контрольный аудит", "разные"],
    ["f9cd2e0dff83371444547709687b58b144f4dff7a25967e12adf7154518c7d38", "f80b2bc5dfde0c86d4ee756fdbf5f9af0f51191fae3cd3dd8982d5e87fe40ed6", "контрольный аудит", "разные"],
    ["3ec9a6dba6c0f7a1b8baa164492eb49b541803b7dd1ec0c28179e2219955cb1e", "274060c075da3d6a67baecf2415024e34d12488dcf36ebd6c7ded9cd8dac8b0f", "контрольный аудит", "разные"],
    ["ce86fe42673759801002d1f3c81e5e94df6904556ef1fb49d69592b99afc75ce", "cf90f9daaef79d9f46f0e34f24cb00808ef2a0a1d1bbc2ed83333f92bc4c7b48", "контрольный аудит", "разные"],
]
if AUDIT_FILE.is_file():
    SAVED_AUDIT = json.loads(AUDIT_FILE.read_text(encoding="utf-8-sig"))
    print(f"Аудит из файла {AUDIT_FILE.resolve()}: {len(SAVED_AUDIT)} пар")
else:
    print(f"Встроенный аудит: {len(SAVED_AUDIT)} пар")

In [ ]:
from time import perf_counter

started = perf_counter()
prepared = prepare_e5(EXPORT_FILES, E5_NPY, E5_SNAPSHOT, top_k=TOP_K,
                      require_full_coverage=REQUIRE_FULL_COVERAGE)
print(f"Карточек с вектором: {len(prepared.dense[1])} из {len(prepared.corpus.cards)}")
print(f"Найдено пар E5: {len(prepared.pool)}")
print(f"Время расчёта соседей: {perf_counter() - started:.1f} с")
print()
print(f"{'Cosine E5':>10} {'Внутри':>10} {'Между*':>10} {'Всего':>10}")
for cut, inside, across in e5_threshold_counts(prepared):
    print(f"{cut:>10.2f} {inside:>10} {across:>10} {inside + across:>10}")
print("* В том числе пары с незаполненным продуктом.")

In [ ]:
# Настройки можно менять и повторно запускать только эту ячейку.
E5_MIN_COSINE = None
SAME_PRODUCT_ONLY = False
WRITE_REPORT = False

options = dict(
    benchmark_file=BENCHMARK_FILE if BENCHMARK_FILE.is_file() else None,
    saved_audit=SAVED_AUDIT, cutoff=E5_MIN_COSINE, same_product_only=SAME_PRODUCT_ONLY,
)
calibration = evaluate_e5(prepared, **options)
print(f"Пар из файла аудита: {len(SAVED_AUDIT)}")
print(f"Пар с совпавшим текстом и векторами для оценки: {len(calibration['audited'])}")
print(f"Пар без совпадения отпечатков: {calibration['missed_audit']}")
if calibration['cutoff'] is None:
    print("Порог не выбран: проверьте, что в части «подбор» есть оба класса разметки.")
else:
    cut = calibration['cutoff']
    policy = calibration['policy']
    print(f"Порог cosine E5: {cut:.6f}")
    print(f"Правило: {policy}")
    print()
    print("На проверочной выборке при выбранном пороге")
    print(f"{'Метод':<47} {'Пар':>5} {'Найдено':>8} {'Лишних':>8} {'Пропуск':>8} {'F1':>7}")
    for split, method, total, tp, fp, fn, _, _ in calibration['metrics']:
        if split == "проверка":
            f1 = 2 * tp / (2 * tp + fp + fn) if (tp + fp + fn) else 0.
            print(f"{method:<47} {total:>5} {tp:>8} {fp:>8} {fn:>8} {f1:>7.3f}")
    if not any(row[0] == "проверка" for row in calibration['metrics']):
        print("Совпавшей разметки для независимой проверки нет.")
    else:
        for _, method, _, tp, fp, fn, _, _ in e5_metrics(calibration['audited'], .95, splits=('проверка',)):
            if method == E5_INSIDE:
                print(f"Порог 0.95 внутри продукта с проверками — найдено {tp}, лишних {fp}, пропущено {fn}")
if WRITE_REPORT:
    summary = export_e5(prepared, OUTPUT_DIR, **options)
    print(f"Отчёты: {OUTPUT_DIR.resolve()}")